In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP22 - INITIALIZATION
# Malignant Cell Identification
# ============================================================

from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import json
import gc
import sys

SEED = 42
np.random.seed(SEED)

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification')
)

DATA_DIR = ROOT / "Data"
NOTEBOOK_DIR = ROOT / "Notebook"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

for p in [
    DATA_DIR,
    NOTEBOOK_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


INPUT_STEP21 = Path(
    _hcc_path('LIHC_scRNA_Step21_Major_Cell_Annotation/Results/GSE149614_STEP21_MAJOR_CELL_ANNOTATED.h5ad')
)

RAW_COUNTS_INPUT = Path(
    _hcc_path('LIHC_scRNA_Step19_QC/Results/GSE149614_CLEAN_COUNTS.h5ad')
)

assert INPUT_STEP21.exists()
assert RAW_COUNTS_INPUT.exists()

adata = sc.read_h5ad(
    INPUT_STEP21
)

print("=" * 90)
print("STEP22 INITIALIZED")
print("=" * 90)

print("Input:")
print(INPUT_STEP21)

print("\nShape:")
print(adata.shape)

print("\nCells:")
print(f"{adata.n_obs:,}")

print("Genes:")
print(f"{adata.n_vars:,}")

print("\nMajor cell types:")
display(
    adata.obs[
        "major_celltype"
    ]
    .value_counts()
)

In [ ]:
# ============================================================
# STEP22 - PUBLICATION FIGURE STYLE
# ============================================================

NATURE = {
    "red": "#E64B35",
    "cyan": "#4DBBD5",
    "green": "#00A087",
    "navy": "#3C5488",
    "salmon": "#F39B7F",
    "purple": "#8491B4",
    "mint": "#91D1C2",
    "darkred": "#DC0000",
    "brown": "#7E6148",
    "beige": "#B09C85",
    "grey": "#BDBDBD",
    "darkgrey": "#4D4D4D"
}

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "Arial",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def save_publication_figure(
    fig,
    filename,
    figure_dir=FIGURES_DIR,
    dpi=600
):

    png_path = (
        Path(figure_dir) /
        f"{filename}.png"
    )

    pdf_path = (
        Path(figure_dir) /
        f"{filename}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print("Saved:")
    print(png_path)
    print(pdf_path)

In [ ]:
# ============================================================
# STEP22A - DEFINE EPITHELIAL / HEPATIC COMPARTMENT
# ============================================================

EPI_CELLTYPES = [
    "Hepatocyte",
    "Ductal_Epithelial",
    "Atypical_Epithelial"
]


epi_mask = (
    adata.obs[
        "major_celltype"
    ]
    .astype(str)
    .isin(
        EPI_CELLTYPES
    )
)


adata_epi = (
    adata[
        epi_mask
    ]
    .copy()
)


print("=" * 90)
print("STEP22 EPITHELIAL COMPARTMENT")
print("=" * 90)

print(
    "Cells:",
    f"{adata_epi.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_epi.n_vars:,}"
)

print("\nComposition:")

display(
    adata_epi.obs[
        "major_celltype"
    ]
    .value_counts()
)

In [ ]:
# ============================================================
# STEP22A - SOURCE STATUS
# ============================================================

adata_epi.obs[
    "source_status"
] = np.where(

    adata_epi.obs[
        "site"
    ].astype(str)
    == "Normal",

    "Normal_site",

    "Tumor_associated_site"
)


print("=" * 90)
print("SOURCE STATUS")
print("=" * 90)

print("\nSite:")
display(
    adata_epi.obs[
        "site"
    ]
    .value_counts()
)

print("\nSource status:")
display(
    adata_epi.obs[
        "source_status"
    ]
    .value_counts()
)

In [ ]:
# ============================================================
# STEP22A - DEFINE NORMAL HEPATOCYTE REFERENCE CANDIDATES
# ============================================================

adata_epi.obs[
    "normal_hepatocyte_reference_candidate"
] = (

    (
        adata_epi.obs[
            "major_celltype"
        ].astype(str)
        == "Hepatocyte"
    )

    &

    (
        adata_epi.obs[
            "site"
        ].astype(str)
        == "Normal"
    )
)


n_reference = int(
    adata_epi.obs[
        "normal_hepatocyte_reference_candidate"
    ].sum()
)


print("=" * 90)
print("NORMAL HEPATOCYTE REFERENCE CANDIDATES")
print("=" * 90)

print(
    "Cells:",
    f"{n_reference:,}"
)


reference_by_patient = (
    adata_epi.obs.loc[
        adata_epi.obs[
            "normal_hepatocyte_reference_candidate"
        ]
    ]
    .groupby(
        "patient",
        observed=True
    )
    .size()
    .sort_values(
        ascending=False
    )
    .rename(
        "N_reference_cells"
    )
)


print("\nBy patient:")

display(
    reference_by_patient.to_frame()
)


reference_by_patient.to_csv(
    RESULTS_DIR /
    "Step22_NormalHepatocyte_Reference_ByPatient.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22A - COMPOSITION AUDIT
# Celltype × Site / Patient / Sample
# ============================================================

# ------------------------------------------------------------
# Major celltype × site
# ------------------------------------------------------------

celltype_site_counts = pd.crosstab(
    adata_epi.obs[
        "major_celltype"
    ],
    adata_epi.obs[
        "site"
    ]
)

celltype_site_pct = (
    pd.crosstab(
        adata_epi.obs[
            "major_celltype"
        ],
        adata_epi.obs[
            "site"
        ],
        normalize="index"
    )
    * 100
)


# ------------------------------------------------------------
# Major celltype × patient
# ------------------------------------------------------------

celltype_patient_counts = pd.crosstab(
    adata_epi.obs[
        "major_celltype"
    ],
    adata_epi.obs[
        "patient"
    ]
)


# ------------------------------------------------------------
# Major celltype × sample
# ------------------------------------------------------------

celltype_sample_counts = pd.crosstab(
    adata_epi.obs[
        "major_celltype"
    ],
    adata_epi.obs[
        "sample"
    ]
)


print("=" * 100)
print("EPITHELIAL COMPARTMENT COMPOSITION")
print("=" * 100)

print("\n[1] Celltype × Site counts")
display(
    celltype_site_counts
)

print("\n[2] Celltype × Site (%)")
display(
    celltype_site_pct.round(2)
)

print("\n[3] Celltype × Patient")
display(
    celltype_patient_counts
)

print("\n[4] Celltype × Sample")
display(
    celltype_sample_counts
)


celltype_site_counts.to_csv(
    RESULTS_DIR /
    "Step22_Epithelial_Celltype_by_Site_Counts.csv",
    encoding="utf-8-sig"
)

celltype_site_pct.to_csv(
    RESULTS_DIR /
    "Step22_Epithelial_Celltype_by_Site_Percent.csv",
    encoding="utf-8-sig"
)

celltype_patient_counts.to_csv(
    RESULTS_DIR /
    "Step22_Epithelial_Celltype_by_Patient.csv",
    encoding="utf-8-sig"
)

celltype_sample_counts.to_csv(
    RESULTS_DIR /
    "Step22_Epithelial_Celltype_by_Sample.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22A - EPITHELIAL BIOLOGY MARKER PANEL
# ============================================================

EPI_MARKERS = {

    "成熟肝细胞": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "HPD",
        "CYP2E1",
        "CPS1"
    ],

    "胆管_祖细胞样": [
        "EPCAM",
        "KRT19",
        "KRT7",
        "KRT8",
        "KRT18",
        "SOX9",
        "CFTR",
        "TACSTD2"
    ],

    "肿瘤相关_胎儿样": [
        "AFP",
        "GPC3",
        "DLK1",
        "IGF2",
        "CLDN6",
        "PRAME",
        "SPP1"
    ],

    "癌睾抗原相关": [
        "PAGE1",
        "PAGE2",
        "GAGE12H",
        "GAGE13",
        "SSX3",
        "PRAME"
    ],

    "增殖": [
        "MKI67",
        "TOP2A",
        "CENPF",
        "CCNB1",
        "CDC20",
        "PRR11"
    ]
}


epi_markers_detected = {

    group: [
        g
        for g in genes
        if g in adata_epi.var_names
    ]

    for group, genes
    in EPI_MARKERS.items()
}


print("Detected markers:")

for group, genes in epi_markers_detected.items():

    print(
        group,
        "→",
        genes
    )

In [ ]:
# ============================================================
# STEP22A - EPITHELIAL MARKER DOTPLOT
# ============================================================

dp = sc.pl.dotplot(

    adata_epi,

    var_names=
    epi_markers_detected,

    groupby=
    "major_celltype",

    standard_scale=
    "var",

    cmap=
    "RdBu_r",

    dot_max=
    0.8,

    figsize=
    (13, 5),

    show=False,

    return_fig=True
)


fig = (
    dp.get_axes()[
        "mainplot_ax"
    ]
    .figure
)


fig.savefig(
    FIGURES_DIR /
    "Step22_01_Epithelial_Biology_DotPlot.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIGURES_DIR /
    "Step22_01_Epithelial_Biology_DotPlot.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP22A - SAVE LOGNORMALIZED EPITHELIAL COMPARTMENT
# ============================================================

EPI_LOGNORM_H5AD = (
    DATA_DIR /
    "GSE149614_STEP22_EPI_LOGNORM.h5ad"
)


adata_epi.write_h5ad(
    EPI_LOGNORM_H5AD,
    compression="gzip"
)


print("=" * 90)
print("EPITHELIAL LOGNORMALIZED OBJECT SAVED")
print("=" * 90)

print(EPI_LOGNORM_H5AD)

print(
    "\nShape:",
    adata_epi.shape
)

In [ ]:
# ============================================================
# STEP22A - SAVE MATCHED RAW COUNTS
# ============================================================

print("=" * 90)
print("LOADING STEP19 RAW COUNTS")
print("=" * 90)

adata_raw_all = sc.read_h5ad(
    RAW_COUNTS_INPUT
)


epi_cells = (
    adata_epi.obs_names
    .astype(str)
)


missing_cells = (
    ~epi_cells.isin(
        adata_raw_all.obs_names
    )
)


print(
    "Missing epithelial cells in raw object:",
    int(
        missing_cells.sum()
    )
)

assert (
    missing_cells.sum()
    == 0
)


adata_epi_raw = (
    adata_raw_all[
        epi_cells
    ]
    .copy()
)


# 把Step21/22最终注释同步进去
cols_to_transfer = [
    "major_cluster_candidate",
    "major_celltype",
    "site",
    "sample",
    "patient"
]


for col in cols_to_transfer:

    if col in adata_epi.obs.columns:

        adata_epi_raw.obs[
            col
        ] = (
            adata_epi.obs.loc[
                adata_epi_raw.obs_names,
                col
            ]
            .values
        )


adata_epi_raw.obs[
    "source_status"
] = (
    adata_epi.obs.loc[
        adata_epi_raw.obs_names,
        "source_status"
    ]
    .values
)


adata_epi_raw.obs[
    "normal_hepatocyte_reference_candidate"
] = (
    adata_epi.obs.loc[
        adata_epi_raw.obs_names,
        "normal_hepatocyte_reference_candidate"
    ]
    .values
)


EPI_RAW_H5AD = (
    DATA_DIR /
    "GSE149614_STEP22_EPI_RAW_COUNTS.h5ad"
)


adata_epi_raw.write_h5ad(
    EPI_RAW_H5AD,
    compression="gzip"
)


print("=" * 90)
print("EPITHELIAL RAW COUNTS SAVED")
print("=" * 90)

print(EPI_RAW_H5AD)

print(
    "\nShape:",
    adata_epi_raw.shape
)


# 释放完整71k raw对象内存
del adata_raw_all

gc.collect()

In [ ]:
# ============================================================
# STEP22A - SAVE COMPARTMENT DECISION
# ============================================================

step22a_decision = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step22A_Epithelial_Compartment",

    "Purpose":
        (
            "Construct the hepatocyte/epithelial compartment "
            "for subsequent malignant-cell identification."
        ),

    "Included_major_celltypes": [
        "Hepatocyte",
        "Ductal_Epithelial",
        "Atypical_Epithelial"
    ],

    "N_cells":
        int(
            adata_epi.n_obs
        ),

    "N_genes":
        int(
            adata_epi.n_vars
        ),

    "Malignant_labels_assigned":
        False,

    "Primary_normal_reference_candidate":
        (
            "Normal-site mature Hepatocyte"
        ),

    "Interpretation_rule":
        (
            "Tumor/PVTT/Lymph origin or atypical epithelial "
            "annotation alone is insufficient to call a cell malignant."
        ),

    "Saved_log_normalized_object":
        str(
            EPI_LOGNORM_H5AD
        ),

    "Saved_raw_counts_object":
        str(
            EPI_RAW_H5AD
        ),

    "Next":
        (
            "Step22B: genomic-coordinate annotation "
            "and CNV inference."
        )
}


DECISION_PATH = (
    LOGS_DIR /
    "Step22A_Compartment_Decision.json"
)


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step22a_decision,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    json.dumps(
        step22a_decision,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP22A - FINAL REVIEW OUTPUT
# 一次性输出进入CNV前必须确认的数据
# ============================================================

print("=" * 110)
print("STEP22A 上皮/肝细胞研究队列最终审计")
print("=" * 110)


# ============================================================
# 1. 总体组成
# ============================================================

print("\n[1] 上皮 / 肝细胞总体组成")
print("-" * 110)

epi_composition = (
    adata_epi.obs["major_celltype"]
    .value_counts()
    .rename_axis("CellType")
    .reset_index(name="N_cells")
)

epi_composition["Pct_cells"] = (
    epi_composition["N_cells"]
    / adata_epi.n_obs
    * 100
)

display(
    epi_composition.round(2)
)

print(
    f"Total epithelial/hepatic cells: "
    f"{adata_epi.n_obs:,}"
)


# ============================================================
# 2. Celltype × Site
# ============================================================

print("\n[2] 各细胞类型来自哪些组织部位：计数")
print("-" * 110)

celltype_site_counts = pd.crosstab(
    adata_epi.obs["major_celltype"],
    adata_epi.obs["site"]
)

display(
    celltype_site_counts
)


print("\n[3] 各细胞类型来自哪些组织部位：行百分比 (%)")
print("-" * 110)

celltype_site_pct = (
    pd.crosstab(
        adata_epi.obs["major_celltype"],
        adata_epi.obs["site"],
        normalize="index"
    )
    * 100
)

display(
    celltype_site_pct.round(2)
)


# ============================================================
# 3. Normal Hepatocyte reference
# ============================================================

print("\n[4] Normal-site Hepatocyte 正常参考细胞")
print("-" * 110)

ref_mask = (
    adata_epi.obs[
        "normal_hepatocyte_reference_candidate"
    ]
)

print(
    "Reference cells:",
    f"{int(ref_mask.sum()):,}"
)


reference_by_patient = (
    adata_epi.obs.loc[
        ref_mask
    ]
    .groupby(
        "patient",
        observed=True
    )
    .size()
    .sort_values(
        ascending=False
    )
    .rename("N_reference_cells")
    .reset_index()
)

display(
    reference_by_patient
)


# ============================================================
# 4. 正常参考来自哪些具体sample
# ============================================================

print("\n[5] Normal-site Hepatocyte 正常参考细胞：按Sample")
print("-" * 110)

reference_by_sample = (
    adata_epi.obs.loc[
        ref_mask
    ]
    .groupby(
        ["patient", "sample"],
        observed=True
    )
    .size()
    .rename("N_reference_cells")
    .reset_index()
    .sort_values(
        "N_reference_cells",
        ascending=False
    )
)

display(
    reference_by_sample
)


# ============================================================
# 5. 三类细胞 × 患者组成
# ============================================================

print("\n[6] 三类细胞在不同患者中的分布")
print("-" * 110)

celltype_patient_counts = pd.crosstab(
    adata_epi.obs["major_celltype"],
    adata_epi.obs["patient"]
)

display(
    celltype_patient_counts
)


# ============================================================
# 6. Atypical epithelial 的来源重点审计
# ============================================================

print("\n[7] Atypical_Epithelial 来源")
print("-" * 110)

atypical_mask = (
    adata_epi.obs["major_celltype"]
    .astype(str)
    == "Atypical_Epithelial"
)

atypical_site = (
    adata_epi.obs.loc[
        atypical_mask,
        "site"
    ]
    .value_counts()
    .rename_axis("Site")
    .reset_index(name="N_cells")
)

atypical_site["Pct"] = (
    atypical_site["N_cells"]
    / atypical_site["N_cells"].sum()
    * 100
)

display(
    atypical_site.round(2)
)


atypical_sample = (
    adata_epi.obs.loc[
        atypical_mask,
        "sample"
    ]
    .value_counts()
    .rename_axis("Sample")
    .reset_index(name="N_cells")
)

atypical_sample["Pct"] = (
    atypical_sample["N_cells"]
    / atypical_sample["N_cells"].sum()
    * 100
)

print("\nAtypical_Epithelial by sample:")

display(
    atypical_sample.round(2)
)


# ============================================================
# 7. 保存这一轮审计结果
# ============================================================

epi_composition.to_csv(
    RESULTS_DIR /
    "Step22A_Epithelial_Composition.csv",
    index=False,
    encoding="utf-8-sig"
)

celltype_site_counts.to_csv(
    RESULTS_DIR /
    "Step22A_Celltype_by_Site_Counts.csv",
    encoding="utf-8-sig"
)

celltype_site_pct.to_csv(
    RESULTS_DIR /
    "Step22A_Celltype_by_Site_Percent.csv",
    encoding="utf-8-sig"
)

reference_by_patient.to_csv(
    RESULTS_DIR /
    "Step22A_NormalReference_ByPatient.csv",
    index=False,
    encoding="utf-8-sig"
)

reference_by_sample.to_csv(
    RESULTS_DIR /
    "Step22A_NormalReference_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

celltype_patient_counts.to_csv(
    RESULTS_DIR /
    "Step22A_Celltype_by_Patient.csv",
    encoding="utf-8-sig"
)

atypical_site.to_csv(
    RESULTS_DIR /
    "Step22A_AtypicalEpithelial_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

atypical_sample.to_csv(
    RESULTS_DIR /
    "Step22A_AtypicalEpithelial_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 110)
print("STEP22A REVIEW OUTPUT COMPLETE")
print("=" * 110)

print(
    "\n确认完这些结果后，"
    "再进入 Step22B：CNV 推断。"
)

In [ ]:
# ============================================================
# STEP22A - FINAL SUMMARY
# ============================================================

import json

step22a_final_summary = {

    "Dataset": "GSE149614",

    "Step": "Step22A_Epithelial_Compartment",

    "Status": "COMPLETE",

    "Total_epithelial_hepatic_cells": 22014,

    "Celltype_counts": {
        "Atypical_Epithelial": 12478,
        "Hepatocyte": 9182,
        "Ductal_Epithelial": 354
    },

    "Celltype_percent": {
        "Atypical_Epithelial": 56.68,
        "Hepatocyte": 41.71,
        "Ductal_Epithelial": 1.61
    },

    "Atypical_epithelial_site_percent": {
        "Tumor": 59.01,
        "PVTT": 28.15,
        "Lymph": 12.53,
        "Normal": 0.30
    },

    "Ductal_epithelial_normal_percent": 95.76,

    "Hepatocyte_normal_percent": 17.25,

    "Normal_hepatocyte_reference_cells": 1584,

    "Normal_reference_note":
        (
            "Normal hepatocyte reference spans 8 patients but is "
            "imbalanced, with HCC03 and HCC04 contributing most cells. "
            "Primary CNV analysis will use all reference cells; "
            "patient-balanced/capped reference will be considered "
            "as a sensitivity analysis."
        ),

    "Atypical_epithelial_interpretation":
        (
            "Strongly enriched in Tumor/PVTT/Lymph and nearly absent "
            "from Normal tissue, supporting a tumor-associated epithelial "
            "identity. Malignancy is not yet assigned."
        ),

    "Key_methodological_rule":
        (
            "Tumor-associated origin, atypical epithelial phenotype, "
            "or tumor-marker expression alone is insufficient to classify "
            "a cell as malignant."
        ),

    "Next_step":
        "Step22B_CNV_Inference"
}


STEP22A_SUMMARY_PATH = (
    LOGS_DIR /
    "Step22A_Final_Summary.json"
)

with open(
    STEP22A_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step22a_final_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


print("=" * 90)
print("STEP22A STATUS = COMPLETE")
print("=" * 90)

print(
    json.dumps(
        step22a_final_summary,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP22B - CNV PREREQUISITE AUDIT
# ============================================================

import importlib.util
import inspect

print("=" * 100)
print("STEP22B CNV 前置条件审计")
print("=" * 100)


# ============================================================
# 1. infercnvpy 是否安装
# ============================================================

infercnvpy_available = (
    importlib.util.find_spec(
        "infercnvpy"
    )
    is not None
)

print("\n[1] infercnvpy")
print("-" * 100)

print(
    "Status:",
    "OK"
    if infercnvpy_available
    else "MISSING"
)


# ============================================================
# 2. 如果已安装，查看真实版本和API
# ============================================================

if infercnvpy_available:

    import infercnvpy as cnv

    print(
        "Version:",
        getattr(
            cnv,
            "__version__",
            "Unknown"
        )
    )

    print("\nCNV-related functions:")

    if hasattr(cnv, "tl"):

        tl_functions = [
            x
            for x in dir(cnv.tl)
            if (
                "cnv" in x.lower()
                or
                "infer" in x.lower()
            )
        ]

        print(
            "cnv.tl:",
            tl_functions
        )

        if hasattr(
            cnv.tl,
            "infercnv"
        ):

            print(
                "\ninfercnv signature:"
            )

            print(
                inspect.signature(
                    cnv.tl.infercnv
                )
            )

        if hasattr(
            cnv.tl,
            "cnv_score"
        ):

            print(
                "\ncnv_score signature:"
            )

            print(
                inspect.signature(
                    cnv.tl.cnv_score
                )
            )


    if hasattr(cnv, "io"):

        io_candidates = [
            x
            for x in dir(cnv.io)
            if (
                "genom" in x.lower()
                or
                "gtf" in x.lower()
                or
                "position" in x.lower()
            )
        ]

        print(
            "\nPossible genomic annotation functions:"
        )

        print(
            io_candidates
        )


# ============================================================
# 3. 检查当前 raw object
# ============================================================

print("\n[2] RAW EPITHELIAL OBJECT")
print("-" * 100)

print(
    "Shape:",
    adata_epi_raw.shape
)

print(
    "Cells:",
    f"{adata_epi_raw.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_epi_raw.n_vars:,}"
)


# ============================================================
# 4. 检查var信息
# ============================================================

print("\n[3] Gene metadata columns")
print("-" * 100)

print(
    adata_epi_raw.var.columns.tolist()
)


# ============================================================
# 5. 检查基因坐标
# ============================================================

possible_chr_cols = [
    "chromosome",
    "chr",
    "chrom",
    "chromosome_name"
]

possible_start_cols = [
    "start",
    "gene_start",
    "start_position"
]

possible_end_cols = [
    "end",
    "gene_end",
    "end_position"
]


chr_found = [
    x
    for x in possible_chr_cols
    if x in adata_epi_raw.var.columns
]

start_found = [
    x
    for x in possible_start_cols
    if x in adata_epi_raw.var.columns
]

end_found = [
    x
    for x in possible_end_cols
    if x in adata_epi_raw.var.columns
]


print(
    "Chromosome column:",
    chr_found
)

print(
    "Start column:",
    start_found
)

print(
    "End column:",
    end_found
)


# ============================================================
# 6. Gene name examples
# ============================================================

print("\n[4] Gene name examples")
print("-" * 100)

print(
    adata_epi_raw.var_names[
        :30
    ].tolist()
)


# ============================================================
# 7. 检查典型基因symbol
# ============================================================

test_genes = [
    "TP53",
    "MYC",
    "ALB",
    "AFP",
    "GPC3",
    "EPCAM",
    "KRT19"
]

print("\n[5] Gene symbol check")
print("-" * 100)

for gene in test_genes:

    print(
        gene,
        "YES"
        if gene in adata_epi_raw.var_names
        else "NO"
    )


# ============================================================
# 8. Normal reference check
# ============================================================

print("\n[6] Normal hepatocyte reference")
print("-" * 100)

if (
    "normal_hepatocyte_reference_candidate"
    in adata_epi_raw.obs.columns
):

    n_ref = int(
        adata_epi_raw.obs[
            "normal_hepatocyte_reference_candidate"
        ].sum()
    )

    print(
        "Reference cells:",
        f"{n_ref:,}"
    )

else:

    print(
        "Reference flag missing."
    )


# ============================================================
# Final status
# ============================================================

print("\n" + "=" * 100)

if (
    infercnvpy_available
    and
    len(chr_found) > 0
    and
    len(start_found) > 0
):

    print(
        "CNV前置条件基本齐全。"
    )

else:

    print(
        "CNV正式运行前还需要补充环境或基因组坐标。"
    )

print("=" * 100)

In [ ]:
# ============================================================
# STEP22B - SAVE ENVIRONMENT BEFORE infercnvpy INSTALLATION
# ============================================================

import sys
import subprocess
from pathlib import Path

ENV_BEFORE = (
    LOGS_DIR /
    "Step22B_PipFreeze_Before_infercnvpy.txt"
)

freeze_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze"
    ],
    capture_output=True,
    text=True,
    check=True
)

ENV_BEFORE.write_text(
    freeze_result.stdout,
    encoding="utf-8"
)

print("=" * 90)
print("CURRENT ENVIRONMENT SAVED")
print("=" * 90)

print("Python:")
print(sys.executable)

print("\nSaved:")
print(ENV_BEFORE)

In [ ]:
# ============================================================
# STEP22B - CELL 14
# PARSE ENSEMBL RELEASE 84 GTF
# ============================================================

from pathlib import Path
import gzip
import re
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Reference directory
# ------------------------------------------------------------

REFERENCE_DIR = (
    ROOT /
    "Reference"
)

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GTF_GZ = (
    REFERENCE_DIR /
    "Homo_sapiens.GRCh38.84.gtf.gz"
)

assert GTF_GZ.exists(), (
    f"GTF file not found:\n{GTF_GZ}"
)

print("=" * 100)
print("STEP22B - PARSE ENSEMBL84 GTF")
print("=" * 100)

print("GTF:")
print(GTF_GZ)

print(
    "\nFile size:",
    f"{GTF_GZ.stat().st_size / 1024**2:.2f} MB"
)


# ------------------------------------------------------------
# GTF attribute parser
# ------------------------------------------------------------

def get_gtf_attribute(
    attributes,
    key
):
    """
    Extract value from GTF attributes.

    Example:
    gene_id "ENSG00000187634";
    gene_name "SAMD11";
    gene_biotype "protein_coding";
    """

    pattern = rf'{key} "([^"]+)"'

    match = re.search(
        pattern,
        attributes
    )

    if match:
        return match.group(1)

    return None


# ------------------------------------------------------------
# Parse gene records
# ------------------------------------------------------------

gene_rows = []


with gzip.open(
    GTF_GZ,
    mode="rt",
    encoding="utf-8"
) as f:

    for line in f:

        # Skip header
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) != 9:
            continue

        chromosome = fields[0]
        feature = fields[2]

        # Only gene-level records
        if feature != "gene":
            continue

        start = int(
            fields[3]
        )

        end = int(
            fields[4]
        )

        strand = fields[6]

        attributes = fields[8]

        gene_id = get_gtf_attribute(
            attributes,
            "gene_id"
        )

        gene_name = get_gtf_attribute(
            attributes,
            "gene_name"
        )

        gene_biotype = get_gtf_attribute(
            attributes,
            "gene_biotype"
        )

        # Some releases may use gene_type instead
        if gene_biotype is None:

            gene_biotype = get_gtf_attribute(
                attributes,
                "gene_type"
            )

        if gene_name is None:
            continue

        gene_rows.append({

            "gene_name":
                gene_name,

            "gene_id":
                gene_id,

            "chromosome_original":
                chromosome,

            "start":
                start,

            "end":
                end,

            "strand":
                strand,

            "gene_biotype":
                gene_biotype
        })


gtf_genes = pd.DataFrame(
    gene_rows
)


print("\nGene records parsed:")
print(
    f"{len(gtf_genes):,}"
)

print("\nUnique gene symbols:")
print(
    f"{gtf_genes['gene_name'].nunique():,}"
)

print("\nExample:")
display(
    gtf_genes.head(20)
)


# ------------------------------------------------------------
# Save raw parsed gene table
# ------------------------------------------------------------

GTF_PARSED_PATH = (
    REFERENCE_DIR /
    "Ensembl84_GRCh38_GeneRecords_Parsed.csv"
)

gtf_genes.to_csv(
    GTF_PARSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(GTF_PARSED_PATH)

In [ ]:
# ============================================================
# STEP22B - CELL 15
# CLEAN GENOMIC COORDINATES
# ============================================================

print("=" * 100)
print("STEP22B - CLEAN GENOMIC COORDINATES")
print("=" * 100)


coords = (
    gtf_genes
    .copy()
)


# ------------------------------------------------------------
# 1. Only canonical chromosomes
# ------------------------------------------------------------

valid_chromosomes = (
    [str(i) for i in range(1, 23)]
    +
    ["X", "Y"]
)


coords = (
    coords[
        coords[
            "chromosome_original"
        ].astype(str)
        .isin(
            valid_chromosomes
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Standard chromosome naming
# ------------------------------------------------------------

coords[
    "chromosome"
] = (
    "chr"
    +
    coords[
        "chromosome_original"
    ].astype(str)
)


print(
    "Gene records on canonical chromosomes:",
    f"{len(coords):,}"
)


# ------------------------------------------------------------
# 3. Check duplicate symbols
# ------------------------------------------------------------

symbol_counts = (
    coords[
        "gene_name"
    ]
    .value_counts()
)


duplicated_symbols = (
    symbol_counts[
        symbol_counts > 1
    ]
)


print(
    "Gene symbols with >1 record:",
    f"{len(duplicated_symbols):,}"
)


# ------------------------------------------------------------
# 4. Detect symbols assigned to >1 chromosome
# ------------------------------------------------------------

n_chr_per_gene = (
    coords
    .groupby(
        "gene_name",
        observed=True
    )[
        "chromosome"
    ]
    .nunique()
)


ambiguous_gene_names = (
    n_chr_per_gene[
        n_chr_per_gene > 1
    ]
    .index
    .tolist()
)


print(
    "Gene symbols mapped to >1 chromosome:",
    f"{len(ambiguous_gene_names):,}"
)


# Save ambiguous genes
ambiguous_gene_table = (
    coords[
        coords[
            "gene_name"
        ].isin(
            ambiguous_gene_names
        )
    ]
    .sort_values(
        [
            "gene_name",
            "chromosome",
            "start"
        ]
    )
    .copy()
)


AMBIGUOUS_PATH = (
    REFERENCE_DIR /
    "Ensembl84_Ambiguous_GeneSymbols_MultiChromosome.csv"
)

ambiguous_gene_table.to_csv(
    AMBIGUOUS_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 5. Remove cross-chromosome ambiguous symbols
#
# Multiple records on SAME chromosome:
# use minimum start and maximum end
# ------------------------------------------------------------

coords_unambiguous = (
    coords[
        ~coords[
            "gene_name"
        ].isin(
            ambiguous_gene_names
        )
    ]
    .copy()
)


coords_clean = (
    coords_unambiguous
    .groupby(
        [
            "gene_name",
            "chromosome"
        ],
        as_index=False,
        observed=True
    )
    .agg(

        start=(
            "start",
            "min"
        ),

        end=(
            "end",
            "max"
        ),

        gene_id=(
            "gene_id",
            "first"
        ),

        gene_biotype=(
            "gene_biotype",
            "first"
        ),

        strand=(
            "strand",
            "first"
        )
    )
)


# ------------------------------------------------------------
# 6. Basic validity check
# ------------------------------------------------------------

assert (
    coords_clean[
        "start"
    ] <=
    coords_clean[
        "end"
    ]
).all()


assert (
    coords_clean[
        "gene_name"
    ]
    .duplicated()
    .sum()
    == 0
)


print(
    "\nFinal clean unique gene coordinates:",
    f"{len(coords_clean):,}"
)


print("\nChromosomes:")

print(
    coords_clean[
        "chromosome"
    ]
    .value_counts()
    .sort_index()
)


print("\nExample:")

display(
    coords_clean.head(20)
)


# ------------------------------------------------------------
# Save clean coordinate reference
# ------------------------------------------------------------

CLEAN_COORD_PATH = (
    REFERENCE_DIR /
    "Ensembl84_GRCh38_GeneCoordinates_Clean.csv"
)

coords_clean.to_csv(
    CLEAN_COORD_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nSaved clean coordinates:")
print(CLEAN_COORD_PATH)

print("\nSaved ambiguous coordinates:")
print(AMBIGUOUS_PATH)

In [ ]:
# ============================================================
# STEP22B - CELL 16
# MATCH GENOMIC COORDINATES TO DATASET GENES
# ============================================================

print("=" * 100)
print("STEP22B - MATCH GENES TO GENOMIC COORDINATES")
print("=" * 100)


# ------------------------------------------------------------
# 1. Build lookup table
# ------------------------------------------------------------

coord_map = (
    coords_clean
    .set_index(
        "gene_name"
    )
)


# ------------------------------------------------------------
# 2. Create dataset annotation table
# ------------------------------------------------------------

gene_annotation = pd.DataFrame(
    index=
    adata_epi.var_names.astype(str)
)

gene_annotation.index.name = (
    "gene_name"
)


# ------------------------------------------------------------
# 3. Map coordinates
# ------------------------------------------------------------

for col in [
    "chromosome",
    "start",
    "end",
    "gene_id",
    "gene_biotype",
    "strand"
]:

    gene_annotation[
        col
    ] = (
        gene_annotation.index
        .map(
            coord_map[col]
        )
    )


# ------------------------------------------------------------
# 4. Availability flag
# ------------------------------------------------------------

gene_annotation[
    "genomic_position_available"
] = (

    gene_annotation[
        "chromosome"
    ].notna()

    &

    gene_annotation[
        "start"
    ].notna()

    &

    gene_annotation[
        "end"
    ].notna()
)


# ------------------------------------------------------------
# 5. Mapping statistics
# ------------------------------------------------------------

n_total = int(
    len(
        gene_annotation
    )
)

n_mapped = int(
    gene_annotation[
        "genomic_position_available"
    ]
    .sum()
)

n_unmapped = (
    n_total -
    n_mapped
)

mapping_pct = (
    n_mapped /
    n_total *
    100
)


print(
    "Total dataset genes:",
    f"{n_total:,}"
)

print(
    "Mapped genes:",
    f"{n_mapped:,}"
)

print(
    "Unmapped genes:",
    f"{n_unmapped:,}"
)

print(
    "Mapping rate:",
    f"{mapping_pct:.2f}%"
)


# ------------------------------------------------------------
# 6. Save mapping summary
# ------------------------------------------------------------

mapping_summary = pd.DataFrame({

    "Metric": [
        "Total_genes",
        "Mapped_genes",
        "Unmapped_genes",
        "Mapping_rate_pct"
    ],

    "Value": [
        n_total,
        n_mapped,
        n_unmapped,
        mapping_pct
    ]
})


MAPPING_SUMMARY_PATH = (
    RESULTS_DIR /
    "Step22B_GeneCoordinate_Mapping_Summary.csv"
)

mapping_summary.to_csv(
    MAPPING_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 7. Save complete annotation table
# ------------------------------------------------------------

FULL_ANNOTATION_PATH = (
    REFERENCE_DIR /
    "GSE149614_GRCh38_Ensembl84_GeneCoordinates_AllGenes.csv"
)

gene_annotation.to_csv(
    FULL_ANNOTATION_PATH,
    encoding="utf-8-sig"
)


print("\nSaved:")
print(MAPPING_SUMMARY_PATH)

print(FULL_ANNOTATION_PATH)

In [ ]:
# ============================================================
# STEP22B - CELL 17
# GENOMIC ANNOTATION QC
# ============================================================

print("=" * 100)
print("STEP22B - GENOMIC ANNOTATION QC")
print("=" * 100)


# ============================================================
# 1. Chromosome distribution
# ============================================================

chrom_order = (
    [f"chr{i}" for i in range(1, 23)]
    +
    ["chrX", "chrY"]
)


chromosome_counts = (
    gene_annotation.loc[
        gene_annotation[
            "genomic_position_available"
        ],
        "chromosome"
    ]
    .value_counts()
    .reindex(
        chrom_order
    )
    .fillna(0)
    .astype(int)
)


chromosome_table = (
    chromosome_counts
    .rename(
        "N_genes"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "Chromosome"
        }
    )
)


print("\n[1] Genes by chromosome")

display(
    chromosome_table
)


# ============================================================
# 2. Important gene coordinate check
# ============================================================

important_genes = [
    "TP53",
    "MYC",
    "ALB",
    "AFP",
    "GPC3",
    "EPCAM",
    "KRT19",
    "CENPA",
    "CCNB1",
    "CDC20",
    "PRR11",
    "MAT1A",
    "CYP7A1"
]


important_gene_coords = (
    gene_annotation
    .reindex(
        important_genes
    )
    .copy()
)


print("\n[2] Important gene coordinates")

display(
    important_gene_coords
)


# ============================================================
# 3. Unmapped genes
# ============================================================

unmapped_genes = (
    gene_annotation[
        ~gene_annotation[
            "genomic_position_available"
        ]
    ]
    .copy()
)


print(
    "\n[3] Number of unmapped genes:",
    f"{len(unmapped_genes):,}"
)


print(
    "\nFirst 100 unmapped genes:"
)

print(
    unmapped_genes
    .index[
        :100
    ]
    .tolist()
)


# ============================================================
# 4. Simple category audit of old-style gene symbols
# ============================================================

unmapped_names = (
    unmapped_genes
    .index
    .astype(str)
)


old_style_patterns = {
    "RP11_":
        unmapped_names.str.startswith(
            "RP11-"
        ).sum(),

    "RP4_":
        unmapped_names.str.startswith(
            "RP4-"
        ).sum(),

    "RP5_":
        unmapped_names.str.startswith(
            "RP5-"
        ).sum(),

    "LINC_":
        unmapped_names.str.startswith(
            "LINC"
        ).sum()
}


unmapped_pattern_table = pd.DataFrame({

    "Pattern":
        list(
            old_style_patterns.keys()
        ),

    "N_unmapped":
        list(
            old_style_patterns.values()
        )
})


print("\n[4] Unmapped legacy-symbol patterns")

display(
    unmapped_pattern_table
)


# ============================================================
# Save
# ============================================================

CHROMOSOME_QC_PATH = (
    RESULTS_DIR /
    "Step22B_Genes_By_Chromosome.csv"
)

IMPORTANT_GENE_PATH = (
    RESULTS_DIR /
    "Step22B_ImportantGene_Coordinates.csv"
)

UNMAPPED_PATH = (
    RESULTS_DIR /
    "Step22B_Unmapped_Genes.csv"
)

UNMAPPED_PATTERN_PATH = (
    RESULTS_DIR /
    "Step22B_UnmappedGene_Patterns.csv"
)


chromosome_table.to_csv(
    CHROMOSOME_QC_PATH,
    index=False,
    encoding="utf-8-sig"
)

important_gene_coords.to_csv(
    IMPORTANT_GENE_PATH,
    encoding="utf-8-sig"
)

unmapped_genes.to_csv(
    UNMAPPED_PATH,
    encoding="utf-8-sig"
)

unmapped_pattern_table.to_csv(
    UNMAPPED_PATTERN_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("\nQC tables saved.")

In [ ]:
# ============================================================
# STEP22B - CELL 18
# ADD GENOMIC POSITIONS TO LOG-NORMALIZED OBJECT
# ============================================================

print("=" * 100)
print("STEP22B - ADD GENOMIC POSITIONS TO AnnData")
print("=" * 100)


# ------------------------------------------------------------
# Ensure exact ordering
# ------------------------------------------------------------

gene_annotation_for_adata = (
    gene_annotation
    .reindex(
        adata_epi.var_names
    )
)


assert (
    gene_annotation_for_adata.index
    ==
    adata_epi.var_names
).all()


# ------------------------------------------------------------
# Write coordinates
# ------------------------------------------------------------

adata_epi.var[
    "chromosome"
] = (
    gene_annotation_for_adata[
        "chromosome"
    ]
    .values
)


# Keep numeric coordinates;
# NaN remains for unmapped genes
adata_epi.var[
    "start"
] = pd.to_numeric(
    gene_annotation_for_adata[
        "start"
    ],
    errors="coerce"
).values


adata_epi.var[
    "end"
] = pd.to_numeric(
    gene_annotation_for_adata[
        "end"
    ],
    errors="coerce"
).values


adata_epi.var[
    "genomic_position_available"
] = (
    gene_annotation_for_adata[
        "genomic_position_available"
    ]
    .astype(bool)
    .values
)


# Optional metadata
adata_epi.var[
    "ensembl84_gene_id"
] = (
    gene_annotation_for_adata[
        "gene_id"
    ]
    .values
)


adata_epi.var[
    "ensembl84_gene_biotype"
] = (
    gene_annotation_for_adata[
        "gene_biotype"
    ]
    .values
)


# ------------------------------------------------------------
# QC
# ------------------------------------------------------------

n_coord = int(
    adata_epi.var[
        "genomic_position_available"
    ]
    .sum()
)


print(
    "AnnData genes:",
    f"{adata_epi.n_vars:,}"
)

print(
    "Genes with genomic coordinates:",
    f"{n_coord:,}"
)

print(
    "Genes without coordinates:",
    f"{adata_epi.n_vars - n_coord:,}"
)


print("\nNew var columns:")

print([
    x
    for x in [
        "chromosome",
        "start",
        "end",
        "genomic_position_available",
        "ensembl84_gene_id",
        "ensembl84_gene_biotype"
    ]
    if x in adata_epi.var.columns
])


print("\nExample:")

display(
    adata_epi.var[
        [
            "chromosome",
            "start",
            "end",
            "genomic_position_available"
        ]
    ]
    .head(20)
)

In [ ]:
# ============================================================
# STEP22B - CELL 19
# SAVE GENOMIC ANNOTATION CHECKPOINT
# ============================================================

import json

print("=" * 100)
print("STEP22B - SAVE GENOMIC ANNOTATION CHECKPOINT")
print("=" * 100)


# ------------------------------------------------------------
# 1. Save fully annotated log-normalized object
# ------------------------------------------------------------

GENOMIC_CHECKPOINT = (
    DATA_DIR /
    "GSE149614_STEP22_EPI_LOGNORM_WITH_GENOMIC_POSITIONS.h5ad"
)


adata_epi.write_h5ad(
    GENOMIC_CHECKPOINT,
    compression="gzip"
)


# ------------------------------------------------------------
# 2. Save final genomic coordinate reference
# ------------------------------------------------------------

FINAL_COORD_PATH = (
    REFERENCE_DIR /
    "GSE149614_GRCh38_Ensembl84_GeneCoordinates.csv"
)


gene_annotation.to_csv(
    FINAL_COORD_PATH,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. Save decision / audit JSON
# ------------------------------------------------------------

genomic_annotation_audit = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step22B_Genomic_Coordinate_Annotation",

    "Reference_genome":
        "GRCh38",

    "Gene_annotation_source":
        "Ensembl release 84",

    "GTF_file":
        str(
            GTF_GZ
        ),

    "Total_dataset_genes":
        int(
            n_total
        ),

    "Genes_with_coordinates":
        int(
            n_mapped
        ),

    "Genes_without_coordinates":
        int(
            n_unmapped
        ),

    "Mapping_rate_pct":
        float(
            mapping_pct
        ),

    "Canonical_chromosomes":
        (
            "chr1-chr22, chrX, chrY"
        ),

    "Cross_chromosome_ambiguous_gene_symbols_removed":
        int(
            len(
                ambiguous_gene_names
            )
        ),

    "Expression_matrix_for_future_CNV":
        (
            "library-size normalized + log1p expression"
        ),

    "Raw_counts_preserved_separately":
        True,

    "Coordinate_table":
        str(
            FINAL_COORD_PATH
        ),

    "Annotated_h5ad":
        str(
            GENOMIC_CHECKPOINT
        ),

    "CNV_inference_performed":
        False,

    "Next_step":
        (
            "Review mapping coverage and genomic annotation QC "
            "before inferCNV calculation."
        )
}


AUDIT_PATH = (
    LOGS_DIR /
    "Step22B_GenomicAnnotation_Audit.json"
)


with open(
    AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        genomic_annotation_audit,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# 4. Output summary
# ------------------------------------------------------------

print(
    "Annotated h5ad:"
)

print(
    GENOMIC_CHECKPOINT
)


print(
    "\nCoordinate table:"
)

print(
    FINAL_COORD_PATH
)


print(
    "\nAudit:"
)

print(
    AUDIT_PATH
)


print("\n" + "=" * 100)

print(
    "Total genes:",
    f"{n_total:,}"
)

print(
    "Mapped genes:",
    f"{n_mapped:,}"
)

print(
    "Unmapped genes:",
    f"{n_unmapped:,}"
)

print(
    "Mapping rate:",
    f"{mapping_pct:.2f}%"
)

print("=" * 100)

print(
    "\nCNV INFERENCE = NOT YET PERFORMED"
)

print(
    "请先检查mapping结果，再进入正式CNV计算。"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP22B - LOAD GENOMIC ANNOTATED OBJECT
# + VERIFY infercnvpy
# ============================================================

from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sys
import importlib.util

GENOMIC_CHECKPOINT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification/Data/GSE149614_STEP22_EPI_LOGNORM_WITH_GENOMIC_POSITIONS.h5ad')
)

assert GENOMIC_CHECKPOINT.exists()

adata_cnv_source = sc.read_h5ad(
    GENOMIC_CHECKPOINT
)

print("=" * 100)
print("STEP22B CNV INPUT")
print("=" * 100)

print(
    "Shape:",
    adata_cnv_source.shape
)

print(
    "Mapped genes:",
    f"{int(adata_cnv_source.var['genomic_position_available'].sum()):,}"
)


# ------------------------------------------------------------
# infercnvpy
# ------------------------------------------------------------

if importlib.util.find_spec(
    "infercnvpy"
) is None:

    raise ImportError(
        "infercnvpy 尚未安装。\n"
        "请运行：\n"
        f"{sys.executable} -m pip install infercnvpy==0.6.1\n"
        "然后重启 Kernel。"
    )

import infercnvpy as cnv

print(
    "\ninfercnvpy version:",
    cnv.__version__
)

print("\nINPUT CHECK = PASS")

In [ ]:
# ============================================================
# STEP22B - CREATE CNV WORKING OBJECT
# ============================================================

mapped_mask = (
    adata_cnv_source.var[
        "genomic_position_available"
    ]
    .fillna(False)
    .values
)

adata_cnv = (
    adata_cnv_source[
        :,
        mapped_mask
    ]
    .copy()
)


# 坐标已经全部非缺失，可以转为整数
adata_cnv.var["start"] = (
    adata_cnv.var["start"]
    .astype(int)
)

adata_cnv.var["end"] = (
    adata_cnv.var["end"]
    .astype(int)
)


print("=" * 100)
print("CNV WORKING OBJECT")
print("=" * 100)

print(
    "Cells:",
    f"{adata_cnv.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_cnv.n_vars:,}"
)

print(
    "Removed unmapped genes:",
    f"{adata_cnv_source.n_vars - adata_cnv.n_vars:,}"
)

assert adata_cnv.n_obs == 22014
assert adata_cnv.n_vars == 25666

print("\nCNV WORKING OBJECT = PASS")

In [ ]:
# ============================================================
# STEP22B - DEFINE CNV REFERENCE CATEGORIES
# ============================================================

adata_cnv.obs[
    "cnv_reference_class"
] = "Query"


# ------------------------------------------------------------
# Normal hepatocyte
# ------------------------------------------------------------

normal_hep_mask = (

    (
        adata_cnv.obs[
            "major_celltype"
        ]
        .astype(str)
        == "Hepatocyte"
    )

    &

    (
        adata_cnv.obs[
            "site"
        ]
        .astype(str)
        == "Normal"
    )
)


adata_cnv.obs.loc[
    normal_hep_mask,
    "cnv_reference_class"
] = "Normal_Hepatocyte"


# ------------------------------------------------------------
# Normal ductal epithelial
# ------------------------------------------------------------

normal_ductal_mask = (

    (
        adata_cnv.obs[
            "major_celltype"
        ]
        .astype(str)
        == "Ductal_Epithelial"
    )

    &

    (
        adata_cnv.obs[
            "site"
        ]
        .astype(str)
        == "Normal"
    )
)


adata_cnv.obs.loc[
    normal_ductal_mask,
    "cnv_reference_class"
] = "Normal_Ductal"


adata_cnv.obs[
    "cnv_reference_class"
] = (
    adata_cnv.obs[
        "cnv_reference_class"
    ]
    .astype("category")
)


reference_counts = (
    adata_cnv.obs[
        "cnv_reference_class"
    ]
    .value_counts()
)


print("=" * 100)
print("CNV REFERENCE DESIGN")
print("=" * 100)

display(
    reference_counts.to_frame(
        "N_cells"
    )
)

print(
    "\nNormal Hepatocyte:",
    int(normal_hep_mask.sum())
)

print(
    "Normal Ductal:",
    int(normal_ductal_mask.sum())
)


assert normal_hep_mask.sum() == 1584
assert normal_ductal_mask.sum() == 339

In [ ]:
# ============================================================
# STEP22B - LOGNORMALIZED INPUT CHECK
# ============================================================

from scipy import sparse

X_test = (
    adata_cnv.X[
        :500,
        :1000
    ]
)

if sparse.issparse(X_test):

    values = X_test.data

else:

    values = (
        np.asarray(X_test)
        .ravel()
    )

values = values[
    values > 0
]


integer_fraction = (
    np.mean(
        np.isclose(
            values,
            np.round(values)
        )
    )
)


print("=" * 100)
print("CNV EXPRESSION INPUT CHECK")
print("=" * 100)

print(
    "Positive-value integer fraction:",
    f"{integer_fraction:.4f}"
)

print(
    "X dtype:",
    adata_cnv.X.dtype
)

print(
    "log1p metadata present:",
    "log1p" in adata_cnv.uns
)


print(
    "\nExpected:"
    "\ninteger fraction should NOT be ~1.0"
    "\nbecause this is log-normalized expression."
)

In [ ]:
# ============================================================
# STEP22B - PRIMARY inferCNV
# ============================================================

import time

print("=" * 100)
print("PRIMARY inferCNV START")
print("=" * 100)

print(
    "Cells:",
    f"{adata_cnv.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_cnv.n_vars:,}"
)

print(
    "\nReference categories:"
)

print(
    "Normal_Hepatocyte"
)

print(
    "Normal_Ductal"
)


start_time = time.time()


cnv.tl.infercnv(

    adata_cnv,

    reference_key=
    "cnv_reference_class",

    reference_cat=[
        "Normal_Hepatocyte",
        "Normal_Ductal"
    ],

    lfc_clip=3,

    window_size=100,

    step=10,

    dynamic_threshold=1.5,

    exclude_chromosomes=(
        "chrX",
        "chrY"
    ),

    chunksize=1000,

    # Windows本地先保守使用2核，避免内存瞬时占用过大
    n_jobs=2,

    inplace=True,

    key_added="cnv",

    calculate_gene_values=False
)


elapsed = (
    time.time()
    -
    start_time
)


print("\n" + "=" * 100)
print("PRIMARY inferCNV COMPLETE")
print("=" * 100)

print(
    "Elapsed minutes:",
    round(
        elapsed / 60,
        2
    )
)

print(
    "CNV representation:",
    adata_cnv.obsm[
        "X_cnv"
    ].shape
)

print(
    "CNV metadata keys:",
    adata_cnv.uns[
        "cnv"
    ].keys()
)

In [ ]:
# ============================================================
# STEP22B - SAVE PRIMARY CNV CHECKPOINT
# ============================================================

CNV_PRIMARY_H5AD = (
    DATA_DIR /
    "GSE149614_STEP22B_PRIMARY_CNV.h5ad"
)


adata_cnv.write_h5ad(
    CNV_PRIMARY_H5AD,
    compression="gzip"
)


print("=" * 100)
print("PRIMARY CNV CHECKPOINT SAVED")
print("=" * 100)

print(
    CNV_PRIMARY_H5AD
)

print(
    "\nShape:",
    adata_cnv.shape
)

print(
    "CNV matrix:",
    adata_cnv.obsm[
        "X_cnv"
    ].shape
)

In [ ]:
# ============================================================
# STEP22B - CELL-LEVEL CNV BURDEN
# Continuous metrics only
# ============================================================

from scipy import sparse


X_cnv = (
    adata_cnv.obsm[
        "X_cnv"
    ]
)


# ------------------------------------------------------------
# Mean absolute CNV
# ------------------------------------------------------------

if sparse.issparse(
    X_cnv
):

    cnv_abs_mean = (
        np.asarray(
            abs(X_cnv)
            .mean(axis=1)
        )
        .ravel()
    )

    cnv_rms = np.sqrt(
        np.asarray(
            X_cnv
            .power(2)
            .mean(axis=1)
        )
        .ravel()
    )

else:

    X_cnv_array = (
        np.asarray(
            X_cnv
        )
    )

    cnv_abs_mean = (
        np.mean(
            np.abs(
                X_cnv_array
            ),
            axis=1
        )
    )

    cnv_rms = np.sqrt(
        np.mean(
            X_cnv_array ** 2,
            axis=1
        )
    )


adata_cnv.obs[
    "cnv_burden_mean_abs"
] = cnv_abs_mean

adata_cnv.obs[
    "cnv_burden_rms"
] = cnv_rms


print("=" * 100)
print("CELL-LEVEL CNV BURDEN")
print("=" * 100)

display(
    adata_cnv.obs[
        [
            "cnv_burden_mean_abs",
            "cnv_burden_rms"
        ]
    ]
    .describe(
        percentiles=[
            .01,
            .05,
            .25,
            .5,
            .75,
            .95,
            .99
        ]
    )
    .T
)

In [ ]:
# ============================================================
# STEP22B - CNV BURDEN AUDIT
# ============================================================

def summarize_cnv(
    df,
    group_col
):

    out = (
        df
        .groupby(
            group_col,
            observed=True
        )
        .agg(

            N_cells=(
                "cnv_burden_mean_abs",
                "size"
            ),

            Mean_CNV=(
                "cnv_burden_mean_abs",
                "mean"
            ),

            Median_CNV=(
                "cnv_burden_mean_abs",
                "median"
            ),

            Q25_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(.25)
            ),

            Q75_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(.75)
            ),

            P95_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(.95)
            )
        )
        .reset_index()
    )

    return out


cnv_by_celltype = summarize_cnv(
    adata_cnv.obs,
    "major_celltype"
)


cnv_by_site = summarize_cnv(
    adata_cnv.obs,
    "site"
)


cnv_by_sample = summarize_cnv(
    adata_cnv.obs,
    "sample"
)


cnv_by_patient = summarize_cnv(
    adata_cnv.obs,
    "patient"
)


cnv_by_reference = summarize_cnv(
    adata_cnv.obs,
    "cnv_reference_class"
)


print("=" * 100)
print("CNV BURDEN BY CELLTYPE")
print("=" * 100)

display(
    cnv_by_celltype.round(4)
)


print("\nCNV BURDEN BY SITE")

display(
    cnv_by_site.round(4)
)


print("\nCNV BURDEN BY REFERENCE STATUS")

display(
    cnv_by_reference.round(4)
)


print("\nCNV BURDEN BY SAMPLE")

display(
    cnv_by_sample.round(4)
)


# ------------------------------------------------------------
# 保存
# ------------------------------------------------------------

cnv_by_celltype.to_csv(
    RESULTS_DIR /
    "Step22B_CNV_Burden_ByCelltype.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_site.to_csv(
    RESULTS_DIR /
    "Step22B_CNV_Burden_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_sample.to_csv(
    RESULTS_DIR /
    "Step22B_CNV_Burden_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_patient.to_csv(
    RESULTS_DIR /
    "Step22B_CNV_Burden_ByPatient.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_reference.to_csv(
    RESULTS_DIR /
    "Step22B_CNV_Burden_ByReference.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22B - SAVE CELL-LEVEL CNV METRICS
# ============================================================

cnv_cell_table = (
    adata_cnv.obs[
        [
            "sample",
            "patient",
            "site",
            "major_celltype",
            "cnv_reference_class",
            "cnv_burden_mean_abs",
            "cnv_burden_rms"
        ]
    ]
    .copy()
)


cnv_cell_table.index.name = (
    "Cell"
)


CNV_CELL_TABLE_PATH = (
    RESULTS_DIR /
    "Step22B_Cell_Level_CNV_Metrics.csv"
)


cnv_cell_table.to_csv(
    CNV_CELL_TABLE_PATH,
    encoding="utf-8-sig"
)


print(
    "Saved:"
)

print(
    CNV_CELL_TABLE_PATH
)

print(
    "\nCells:",
    f"{len(cnv_cell_table):,}"
)

In [ ]:
# ============================================================
# STEP22B - CHROMOSOME CNV SUMMARY
# By major cell type
# ============================================================

axes_dict = (
    cnv.pl.chromosome_heatmap_summary(

        adata_cnv,

        groupby=
        "major_celltype",

        use_rep=
        "cnv",

        cmap=
        "bwr",

        figsize=
        (16, 5),

        show=False
    )
)


fig = None

for ax_obj in axes_dict.values():

    if hasattr(
        ax_obj,
        "figure"
    ):

        fig = ax_obj.figure
        break


assert fig is not None


fig.savefig(
    FIGURES_DIR /
    "Step22B_01_CNV_ChromosomeSummary_ByCelltype.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIGURES_DIR /
    "Step22B_01_CNV_ChromosomeSummary_ByCelltype.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP22B - CHROMOSOME CNV SUMMARY
# By sample
# ============================================================

axes_dict = (
    cnv.pl.chromosome_heatmap_summary(

        adata_cnv,

        groupby=
        "sample",

        use_rep=
        "cnv",

        cmap=
        "bwr",

        figsize=
        (16, 10),

        show=False
    )
)


fig = None

for ax_obj in axes_dict.values():

    if hasattr(
        ax_obj,
        "figure"
    ):

        fig = ax_obj.figure
        break


assert fig is not None


fig.savefig(
    FIGURES_DIR /
    "Step22B_02_CNV_ChromosomeSummary_BySample.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    FIGURES_DIR /
    "Step22B_02_CNV_ChromosomeSummary_BySample.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP22B - SAVE CNV METRICS CHECKPOINT
# ============================================================

CNV_METRICS_H5AD = (
    DATA_DIR /
    "GSE149614_STEP22B_PRIMARY_CNV_WITH_METRICS.h5ad"
)


adata_cnv.write_h5ad(
    CNV_METRICS_H5AD,
    compression="gzip"
)


print("=" * 100)
print("STEP22B CNV METRICS CHECKPOINT SAVED")
print("=" * 100)

print(
    CNV_METRICS_H5AD
)

In [ ]:
# ============================================================
# STEP22B - FINAL REVIEW OUTPUT
# 一次性输出下一步判断所需核心结果
# ============================================================

from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import json


print("=" * 110)
print("STEP22B CNV结果汇总")
print("=" * 110)


# ============================================================
# 0. 如果当前内存里没有 adata_cnv，就读取已保存checkpoint
# ============================================================

if "adata_cnv" not in globals():

    CNV_METRICS_H5AD = Path(
        r"E:\TCGA\LIHC_scRNA_Step22_Malignant_Identification"
        r"\Data\GSE149614_STEP22B_PRIMARY_CNV_WITH_METRICS.h5ad"
    )

    assert CNV_METRICS_H5AD.exists(), (
        f"CNV checkpoint不存在：\n{CNV_METRICS_H5AD}"
    )

    adata_cnv = sc.read_h5ad(
        CNV_METRICS_H5AD
    )

    print(
        "\n已重新读取CNV checkpoint："
    )

    print(
        CNV_METRICS_H5AD
    )


# ============================================================
# 1. CNV矩阵基本状态
# ============================================================

print("\n[1] CNV基本状态")
print("-" * 110)

print(
    "Cells:",
    f"{adata_cnv.n_obs:,}"
)

print(
    "Genes used for CNV:",
    f"{adata_cnv.n_vars:,}"
)

print(
    "CNV representation shape:",
    adata_cnv.obsm["X_cnv"].shape
)

print(
    "Normal Hepatocyte reference:",
    int(
        (
            adata_cnv.obs[
                "cnv_reference_class"
            ].astype(str)
            == "Normal_Hepatocyte"
        ).sum()
    )
)

print(
    "Normal Ductal reference:",
    int(
        (
            adata_cnv.obs[
                "cnv_reference_class"
            ].astype(str)
            == "Normal_Ductal"
        ).sum()
    )
)


# ============================================================
# 2. 全部细胞CNV burden分布
# ============================================================

print("\n[2] 全部细胞 CNV burden 分布")
print("-" * 110)

cnv_global = (
    adata_cnv.obs[
        [
            "cnv_burden_mean_abs",
            "cnv_burden_rms"
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

display(
    cnv_global.round(5)
)


# ============================================================
# 3. 辅助函数
# ============================================================

def cnv_summary(
    adata,
    group_col
):

    result = (
        adata.obs
        .groupby(
            group_col,
            observed=True
        )
        .agg(

            N_cells=(
                "cnv_burden_mean_abs",
                "size"
            ),

            Mean_CNV=(
                "cnv_burden_mean_abs",
                "mean"
            ),

            Median_CNV=(
                "cnv_burden_mean_abs",
                "median"
            ),

            Q25_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(0.25)
            ),

            Q75_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(0.75)
            ),

            P90_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(0.90)
            ),

            P95_CNV=(
                "cnv_burden_mean_abs",
                lambda x:
                x.quantile(0.95)
            ),

            Mean_RMS=(
                "cnv_burden_rms",
                "mean"
            ),

            Median_RMS=(
                "cnv_burden_rms",
                "median"
            )
        )
        .reset_index()
    )

    return result


# ============================================================
# 4. CNV burden by major celltype
# ============================================================

cnv_by_celltype = cnv_summary(
    adata_cnv,
    "major_celltype"
)

print("\n[3] CNV BURDEN BY CELLTYPE")
print("-" * 110)

display(
    cnv_by_celltype.round(5)
)


# ============================================================
# 5. CNV burden by reference status
# ============================================================

cnv_by_reference = cnv_summary(
    adata_cnv,
    "cnv_reference_class"
)

print("\n[4] CNV BURDEN BY REFERENCE STATUS")
print("-" * 110)

display(
    cnv_by_reference.round(5)
)


# ============================================================
# 6. CNV burden by site
# ============================================================

cnv_by_site = cnv_summary(
    adata_cnv,
    "site"
)

print("\n[5] CNV BURDEN BY SITE")
print("-" * 110)

display(
    cnv_by_site.round(5)
)


# ============================================================
# 7. CNV burden by sample
# ============================================================

cnv_by_sample = cnv_summary(
    adata_cnv,
    "sample"
)

cnv_by_sample = (
    cnv_by_sample
    .sort_values(
        "Median_CNV",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n[6] CNV BURDEN BY SAMPLE")
print("-" * 110)

display(
    cnv_by_sample.round(5)
)


# ============================================================
# 8. CNV burden by patient
# ============================================================

cnv_by_patient = cnv_summary(
    adata_cnv,
    "patient"
)

cnv_by_patient = (
    cnv_by_patient
    .sort_values(
        "Median_CNV",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n[7] CNV BURDEN BY PATIENT")
print("-" * 110)

display(
    cnv_by_patient.round(5)
)


# ============================================================
# 9. 最关键：
# major_celltype × site 联合CNV
# ============================================================

cnv_celltype_site = (
    adata_cnv.obs
    .groupby(
        [
            "major_celltype",
            "site"
        ],
        observed=True
    )
    .agg(

        N_cells=(
            "cnv_burden_mean_abs",
            "size"
        ),

        Median_CNV=(
            "cnv_burden_mean_abs",
            "median"
        ),

        Mean_CNV=(
            "cnv_burden_mean_abs",
            "mean"
        ),

        P95_CNV=(
            "cnv_burden_mean_abs",
            lambda x:
            x.quantile(0.95)
        )
    )
    .reset_index()
)

print("\n[8] CELLTYPE × SITE CNV")
print("-" * 110)

display(
    cnv_celltype_site.round(5)
)


# ============================================================
# 10. Atypical epithelial重点看
# ============================================================

atypical_mask = (
    adata_cnv.obs[
        "major_celltype"
    ]
    .astype(str)
    == "Atypical_Epithelial"
)

atypical_cnv_by_sample = (
    adata_cnv.obs.loc[
        atypical_mask
    ]
    .groupby(
        "sample",
        observed=True
    )
    .agg(

        N_cells=(
            "cnv_burden_mean_abs",
            "size"
        ),

        Median_CNV=(
            "cnv_burden_mean_abs",
            "median"
        ),

        Mean_CNV=(
            "cnv_burden_mean_abs",
            "mean"
        ),

        Q25_CNV=(
            "cnv_burden_mean_abs",
            lambda x:
            x.quantile(.25)
        ),

        Q75_CNV=(
            "cnv_burden_mean_abs",
            lambda x:
            x.quantile(.75)
        ),

        P95_CNV=(
            "cnv_burden_mean_abs",
            lambda x:
            x.quantile(.95)
        )
    )
    .reset_index()
    .sort_values(
        "Median_CNV",
        ascending=False
    )
)

print("\n[9] ATYPICAL EPITHELIAL：按Sample的CNV")
print("-" * 110)

display(
    atypical_cnv_by_sample.round(5)
)


# ============================================================
# 11. Tumor-site Hepatocyte重点看
# ============================================================

tumor_hep_mask = (

    (
        adata_cnv.obs[
            "major_celltype"
        ]
        .astype(str)
        == "Hepatocyte"
    )

    &

    (
        adata_cnv.obs[
            "site"
        ]
        .astype(str)
        != "Normal"
    )
)


tumor_hep_by_sample = (
    adata_cnv.obs.loc[
        tumor_hep_mask
    ]
    .groupby(
        "sample",
        observed=True
    )
    .agg(

        N_cells=(
            "cnv_burden_mean_abs",
            "size"
        ),

        Median_CNV=(
            "cnv_burden_mean_abs",
            "median"
        ),

        Mean_CNV=(
            "cnv_burden_mean_abs",
            "mean"
        ),

        P95_CNV=(
            "cnv_burden_mean_abs",
            lambda x:
            x.quantile(.95)
        )
    )
    .reset_index()
    .sort_values(
        "Median_CNV",
        ascending=False
    )
)

print("\n[10] TUMOR-SITE HEPATOCYTE：按Sample的CNV")
print("-" * 110)

display(
    tumor_hep_by_sample.round(5)
)


# ============================================================
# 12. 图像文件检查
# ============================================================

print("\n[11] CNV FIGURE CHECK")
print("-" * 110)

figure_names = [

    "Step22B_01_CNV_ChromosomeSummary_ByCelltype",

    "Step22B_02_CNV_ChromosomeSummary_BySample"
]


for name in figure_names:

    png_path = (
        FIGURES_DIR /
        f"{name}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{name}.pdf"
    )

    print(
        name,
        "| PNG:",
        "OK"
        if png_path.exists()
        else "MISSING",
        "| PDF:",
        "OK"
        if pdf_path.exists()
        else "MISSING"
    )


# ============================================================
# 13. 保存这次review表
# ============================================================

cnv_global.to_csv(
    RESULTS_DIR /
    "Step22B_Review_Global_CNV_Distribution.csv",
    encoding="utf-8-sig"
)

cnv_by_celltype.to_csv(
    RESULTS_DIR /
    "Step22B_Review_CNV_ByCelltype.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_reference.to_csv(
    RESULTS_DIR /
    "Step22B_Review_CNV_ByReference.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_site.to_csv(
    RESULTS_DIR /
    "Step22B_Review_CNV_BySite.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_sample.to_csv(
    RESULTS_DIR /
    "Step22B_Review_CNV_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_by_patient.to_csv(
    RESULTS_DIR /
    "Step22B_Review_CNV_ByPatient.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_celltype_site.to_csv(
    RESULTS_DIR /
    "Step22B_Review_CNV_Celltype_x_Site.csv",
    index=False,
    encoding="utf-8-sig"
)

atypical_cnv_by_sample.to_csv(
    RESULTS_DIR /
    "Step22B_Review_AtypicalEpithelial_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)

tumor_hep_by_sample.to_csv(
    RESULTS_DIR /
    "Step22B_Review_TumorSiteHepatocyte_BySample.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 110)
print("STEP22B REVIEW OUTPUT COMPLETE")
print("=" * 110)

print(
    "\n目前仍未进行 malignant / non-malignant 最终分类。"
)

print(
    "下一步将根据CNV profile、正常参考分布、"
    "患者/样本来源和敏感性分析决定恶性细胞。"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP22C - CELL 32
# 读取CNV对象 + 初始化
# ============================================================

from pathlib import Path
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score

SEED = 42
np.random.seed(SEED)

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

CNV_INPUT = (
    DATA_DIR /
    "GSE149614_STEP22B_PRIMARY_CNV_WITH_METRICS.h5ad"
)

assert CNV_INPUT.exists()

adata_cnv = sc.read_h5ad(
    CNV_INPUT
)

print("=" * 100)
print("STEP22C 初始化")
print("=" * 100)

print(
    "Cells:",
    f"{adata_cnv.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_cnv.n_vars:,}"
)

print(
    "CNV matrix:",
    adata_cnv.obsm["X_cnv"].shape
)

print(
    "CNV burden:",
    "cnv_burden_mean_abs"
    in adata_cnv.obs.columns
)

assert "X_cnv" in adata_cnv.obsm
assert "cnv_burden_mean_abs" in adata_cnv.obs.columns

print("\nSTEP22C INPUT = PASS")

In [ ]:
# ============================================================
# STEP22C - CELL 33
# 绘图规范
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Arial"
]

plt.rcParams["axes.unicode_minus"] = False

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def save_publication_figure(
    fig,
    filename
):

    png_path = (
        FIGURES_DIR /
        f"{filename}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{filename}.pdf"
    )

    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print("Saved:")
    print(png_path)
    print(pdf_path)

In [ ]:
# ============================================================
# STEP22C - CELL 34
# CNV burden 技术混杂审计
# ============================================================

TECHNICAL_VARS = [
    x
    for x in [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt"
    ]
    if x in adata_cnv.obs.columns
]

print("=" * 100)
print("CNV BURDEN 技术混杂审计")
print("=" * 100)

print(
    "Technical variables:",
    TECHNICAL_VARS
)


confound_rows = []


# ------------------------------------------------------------
# 1. 全体细胞
# ------------------------------------------------------------

for var in TECHNICAL_VARS:

    temp = (
        adata_cnv.obs[
            [
                "cnv_burden_mean_abs",
                var
            ]
        ]
        .dropna()
    )

    rho, p = spearmanr(
        temp[
            "cnv_burden_mean_abs"
        ],
        temp[var]
    )

    confound_rows.append({

        "Scope":
            "All_cells",

        "Group":
            "All",

        "Technical_variable":
            var,

        "N_cells":
            len(temp),

        "Spearman_rho":
            rho,

        "P_value":
            p
    })


# ------------------------------------------------------------
# 2. 分 major celltype
#
# 防止整体相关只是因为：
# Atypical本身RNA多且CNV高
# ------------------------------------------------------------

for celltype in (
    adata_cnv.obs[
        "major_celltype"
    ]
    .astype(str)
    .unique()
):

    mask = (
        adata_cnv.obs[
            "major_celltype"
        ]
        .astype(str)
        == celltype
    )

    for var in TECHNICAL_VARS:

        temp = (
            adata_cnv.obs.loc[
                mask,
                [
                    "cnv_burden_mean_abs",
                    var
                ]
            ]
            .dropna()
        )

        if len(temp) < 30:
            continue

        rho, p = spearmanr(
            temp[
                "cnv_burden_mean_abs"
            ],
            temp[var]
        )

        confound_rows.append({

            "Scope":
                "Within_celltype",

            "Group":
                celltype,

            "Technical_variable":
                var,

            "N_cells":
                len(temp),

            "Spearman_rho":
                rho,

            "P_value":
                p
        })


cnv_confound = pd.DataFrame(
    confound_rows
)


print("\n[全体细胞]")

display(
    cnv_confound[
        cnv_confound[
            "Scope"
        ] == "All_cells"
    ]
    .round(5)
)


print("\n[各细胞类型内部]")

display(
    cnv_confound[
        cnv_confound[
            "Scope"
        ] == "Within_celltype"
    ]
    .round(5)
)


cnv_confound.to_csv(
    RESULTS_DIR /
    "Step22C_CNV_TechnicalConfound_Spearman.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C - CELL 35
# CNV burden vs 技术指标
# ============================================================

rng = np.random.default_rng(
    SEED
)

n_plot = min(
    10000,
    adata_cnv.n_obs
)

plot_idx = rng.choice(
    adata_cnv.n_obs,
    size=n_plot,
    replace=False
)


for i, var in enumerate(
    TECHNICAL_VARS,
    start=1
):

    x = (
        adata_cnv.obs[
            var
        ]
        .values[
            plot_idx
        ]
    )

    y = (
        adata_cnv.obs[
            "cnv_burden_mean_abs"
        ]
        .values[
            plot_idx
        ]
    )


    full_temp = (
        adata_cnv.obs[
            [
                var,
                "cnv_burden_mean_abs"
            ]
        ]
        .dropna()
    )

    rho, p = spearmanr(
        full_temp[var],
        full_temp[
            "cnv_burden_mean_abs"
        ]
    )


    fig, ax = plt.subplots(
        figsize=(6.2, 5.2)
    )

    ax.scatter(
        x,
        y,
        s=5,
        alpha=0.25,
        edgecolors="none",
        rasterized=True
    )

    ax.set_xlabel(
        var
    )

    ax.set_ylabel(
        "CNV burden（平均绝对值）"
    )

    ax.set_title(
        f"CNV burden 与 {var}"
    )

    ax.text(
        0.03,
        0.97,
        f"Spearman ρ = {rho:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()

    save_publication_figure(
        fig,
        f"Step22C_0{i}_CNV_Burden_vs_{var}"
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP22C - CELL 36
# CNV PCA
# ============================================================

X_cnv = (
    adata_cnv.obsm[
        "X_cnv"
    ]
)


print("=" * 100)
print("CNV PCA")
print("=" * 100)

print(
    "Input CNV matrix:",
    X_cnv.shape
)


# ------------------------------------------------------------
# 构造临时AnnData
# X = CNV窗口
# ------------------------------------------------------------

adata_cnv_pca = ad.AnnData(

    X=(
        X_cnv.copy()
        if sparse.issparse(X_cnv)
        else np.asarray(
            X_cnv,
            dtype=np.float32
        )
    ),

    obs=
        adata_cnv.obs.copy()
)


# ------------------------------------------------------------
# PCA
# ------------------------------------------------------------

sc.tl.pca(
    adata_cnv_pca,
    n_comps=30,
    zero_center=True,
    svd_solver="arpack",
    random_state=42
)


# ------------------------------------------------------------
# 转回主对象
# ------------------------------------------------------------

adata_cnv.obsm[
    "X_pca_cnv"
] = (
    adata_cnv_pca.obsm[
        "X_pca"
    ]
    .copy()
)


cnv_pca_variance = pd.DataFrame({

    "PC":
        np.arange(
            1,
            31
        ),

    "VarianceRatio":
        adata_cnv_pca.uns[
            "pca"
        ][
            "variance_ratio"
        ]
})


cnv_pca_variance[
    "CumulativeVariance"
] = (
    cnv_pca_variance[
        "VarianceRatio"
    ]
    .cumsum()
)


print(
    "CNV PCA shape:",
    adata_cnv.obsm[
        "X_pca_cnv"
    ].shape
)

display(
    cnv_pca_variance.head(20)
)


cnv_pca_variance.to_csv(
    RESULTS_DIR /
    "Step22C_CNV_PCA_Variance.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C - CELL 37
# CNV邻居网络 + UMAP
# ============================================================

print("=" * 100)
print("CNV NEIGHBOR GRAPH + UMAP")
print("=" * 100)


sc.pp.neighbors(
    adata_cnv,

    use_rep=
    "X_pca_cnv",

    n_neighbors=15,

    n_pcs=20,

    key_added=
    "neighbors_cnv",

    random_state=42
)


sc.tl.umap(
    adata_cnv,

    neighbors_key=
    "neighbors_cnv",

    random_state=42
)


adata_cnv.obsm[
    "X_umap_cnv"
] = (
    adata_cnv.obsm[
        "X_umap"
    ]
    .copy()
)


print(
    "CNV UMAP:",
    adata_cnv.obsm[
        "X_umap_cnv"
    ].shape
)

In [ ]:
# ============================================================
# STEP22C - CELL 38
# CNV UMAP诊断图
# ============================================================

plot_vars = [
    (
        "cnv_reference_class",
        "Step22C_04_CNV_UMAP_ByReference"
    ),
    (
        "major_celltype",
        "Step22C_05_CNV_UMAP_ByCelltype"
    ),
    (
        "site",
        "Step22C_06_CNV_UMAP_BySite"
    ),
    (
        "patient",
        "Step22C_07_CNV_UMAP_ByPatient"
    ),
    (
        "sample",
        "Step22C_08_CNV_UMAP_BySample"
    )
]


for color_col, filename in plot_vars:

    ax = sc.pl.embedding(

        adata_cnv,

        basis=
        "umap_cnv",

        color=
        color_col,

        size=6,

        alpha=0.75,

        frameon=False,

        legend_loc=
        "right margin",

        title=
        f"CNV空间：{color_col}",

        show=False
    )

    fig = ax.figure

    fig.set_size_inches(
        8.2,
        6.4
    )

    fig.tight_layout()

    save_publication_figure(
        fig,
        filename
    )

    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP22C - CELL 39
# CNV空间多分辨率Leiden
# ============================================================

cnv_resolutions = [
    0.2,
    0.4,
    0.6,
    0.8
]


cnv_cluster_summary_rows = []


for res in cnv_resolutions:

    key = (
        "cnv_leiden_r"
        +
        str(res).replace(
            ".",
            "_"
        )
    )

    print(
        f"Running CNV Leiden resolution = {res}"
    )

    sc.tl.leiden(

        adata_cnv,

        resolution=res,

        neighbors_key=
        "neighbors_cnv",

        key_added=key,

        random_state=42,

        flavor="igraph",

        n_iterations=2,

        directed=False
    )


    sizes = (
        adata_cnv.obs[
            key
        ]
        .value_counts()
    )


    cnv_cluster_summary_rows.append({

        "Resolution":
            res,

        "Cluster_column":
            key,

        "N_clusters":
            len(sizes),

        "Smallest_cluster":
            int(
                sizes.min()
            ),

        "Largest_cluster":
            int(
                sizes.max()
            ),

        "Median_cluster_size":
            float(
                sizes.median()
            ),

        "Clusters_lt_100":
            int(
                (
                    sizes < 100
                ).sum()
            )
    })


cnv_cluster_summary = pd.DataFrame(
    cnv_cluster_summary_rows
)


print("\nCNV clustering summary:")

display(
    cnv_cluster_summary
)


cnv_cluster_summary.to_csv(
    RESULTS_DIR /
    "Step22C_CNV_Leiden_Resolution_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C - CELL 40
# CNV聚类稳定性
# ============================================================

cnv_pairs = [
    (0.2, 0.4),
    (0.4, 0.6),
    (0.6, 0.8)
]


cnv_ari_rows = []


for r1, r2 in cnv_pairs:

    key1 = (
        "cnv_leiden_r"
        +
        str(r1).replace(
            ".",
            "_"
        )
    )

    key2 = (
        "cnv_leiden_r"
        +
        str(r2).replace(
            ".",
            "_"
        )
    )


    ari = adjusted_rand_score(

        adata_cnv.obs[
            key1
        ].astype(str),

        adata_cnv.obs[
            key2
        ].astype(str)
    )


    cnv_ari_rows.append({

        "Resolution_1":
            r1,

        "Resolution_2":
            r2,

        "ARI":
            ari
    })


cnv_resolution_ari = pd.DataFrame(
    cnv_ari_rows
)


display(
    cnv_resolution_ari.round(4)
)


cnv_resolution_ari.to_csv(
    RESULTS_DIR /
    "Step22C_CNV_Leiden_ARI.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C - CELL 41
# CNV Leiden UMAP
# ============================================================

for res in cnv_resolutions:

    key = (
        "cnv_leiden_r"
        +
        str(res).replace(
            ".",
            "_"
        )
    )

    ax = sc.pl.embedding(

        adata_cnv,

        basis=
        "umap_cnv",

        color=
        key,

        size=6,

        alpha=0.80,

        frameon=False,

        legend_loc=
        "on data",

        legend_fontsize=8,

        title=
        f"CNV聚类，resolution={res}",

        show=False
    )


    fig = ax.figure

    fig.set_size_inches(
        7.5,
        6.3
    )

    fig.tight_layout()


    save_publication_figure(
        fig,
        (
            "Step22C_CNV_Leiden_"
            f"Resolution_{str(res).replace('.', '_')}"
        )
    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# STEP22C - CELL 42
# CNV Cluster组成审计
# ============================================================

all_cluster_tables = []


for res in cnv_resolutions:

    key = (
        "cnv_leiden_r"
        +
        str(res).replace(
            ".",
            "_"
        )
    )


    clusters = sorted(
        adata_cnv.obs[
            key
        ]
        .astype(str)
        .unique(),
        key=lambda x:
        int(x)
    )


    rows = []


    for cluster in clusters:

        mask = (
            adata_cnv.obs[
                key
            ]
            .astype(str)
            == cluster
        )

        obs_sub = (
            adata_cnv.obs.loc[
                mask
            ]
        )

        n = len(
            obs_sub
        )


        def pct(condition):
            return (
                np.mean(
                    condition
                )
                *
                100
            )


        rows.append({

            "Resolution":
                res,

            "CNV_cluster":
                cluster,

            "N_cells":
                n,

            "Median_CNV":
                obs_sub[
                    "cnv_burden_mean_abs"
                ].median(),

            "Mean_CNV":
                obs_sub[
                    "cnv_burden_mean_abs"
                ].mean(),

            "Pct_Normal_Hep_reference":
                pct(
                    obs_sub[
                        "cnv_reference_class"
                    ]
                    .astype(str)
                    ==
                    "Normal_Hepatocyte"
                ),

            "Pct_Normal_Ductal_reference":
                pct(
                    obs_sub[
                        "cnv_reference_class"
                    ]
                    .astype(str)
                    ==
                    "Normal_Ductal"
                ),

            "Pct_Atypical_Epithelial":
                pct(
                    obs_sub[
                        "major_celltype"
                    ]
                    .astype(str)
                    ==
                    "Atypical_Epithelial"
                ),

            "Pct_Hepatocyte":
                pct(
                    obs_sub[
                        "major_celltype"
                    ]
                    .astype(str)
                    ==
                    "Hepatocyte"
                ),

            "Pct_Ductal_Epithelial":
                pct(
                    obs_sub[
                        "major_celltype"
                    ]
                    .astype(str)
                    ==
                    "Ductal_Epithelial"
                ),

            "Pct_Normal_site":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "Normal"
                ),

            "Pct_Tumor_site":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "Tumor"
                ),

            "Pct_PVTT":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "PVTT"
                ),

            "Pct_Lymph":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "Lymph"
                )
        })


    temp = pd.DataFrame(
        rows
    )

    all_cluster_tables.append(
        temp
    )


cnv_cluster_composition = pd.concat(
    all_cluster_tables,
    ignore_index=True
)


CNV_CLUSTER_COMP_PATH = (
    RESULTS_DIR /
    "Step22C_CNV_Cluster_Composition_AllResolutions.csv"
)


cnv_cluster_composition.to_csv(
    CNV_CLUSTER_COMP_PATH,
    index=False,
    encoding="utf-8-sig"
)


for res in cnv_resolutions:

    print(
        "\n" +
        "=" * 100
    )

    print(
        f"CNV resolution = {res}"
    )

    print(
        "=" * 100
    )

    display(
        cnv_cluster_composition[
            cnv_cluster_composition[
                "Resolution"
            ] == res
        ]
        .round(2)
    )

In [ ]:
# ============================================================
# STEP22C - CELL 43
# 保存CNV空间聚类Checkpoint
# ============================================================

STEP22C_CHECKPOINT = (
    DATA_DIR /
    "GSE149614_STEP22C_CNV_PCA_UMAP_CLUSTERED.h5ad"
)


adata_cnv.write_h5ad(
    STEP22C_CHECKPOINT,
    compression="gzip"
)


print("=" * 100)
print("STEP22C CHECKPOINT SAVED")
print("=" * 100)

print(
    STEP22C_CHECKPOINT
)

In [ ]:
# ============================================================
# STEP22C - CELL 44
# 最终Review输出
# ============================================================

print("=" * 110)
print("STEP22C CNV质量与聚类结果汇总")
print("=" * 110)


# ============================================================
# 1. 技术混杂
# ============================================================

print("\n[1] 全体细胞 CNV burden × 技术指标")
print("-" * 110)

display(
    cnv_confound[
        cnv_confound[
            "Scope"
        ]
        == "All_cells"
    ]
    .round(5)
)


print("\n[2] 各细胞类型内部相关")
print("-" * 110)

display(
    cnv_confound[
        cnv_confound[
            "Scope"
        ]
        == "Within_celltype"
    ]
    .round(5)
)


# ============================================================
# 2. PCA
# ============================================================

print("\n[3] CNV PCA")
print("-" * 110)

print(
    "PCA shape:",
    adata_cnv.obsm[
        "X_pca_cnv"
    ].shape
)

display(
    cnv_pca_variance
    .head(15)
    .round(5)
)


# ============================================================
# 3. CNV clustering
# ============================================================

print("\n[4] CNV不同分辨率")
print("-" * 110)

display(
    cnv_cluster_summary
)


print("\n[5] CNV聚类ARI")
print("-" * 110)

display(
    cnv_resolution_ari
    .round(4)
)


# ============================================================
# 4. 重点显示0.4和0.6
# ============================================================

print("\n[6] Resolution 0.4 Cluster组成")
print("-" * 110)

display(
    cnv_cluster_composition[
        cnv_cluster_composition[
            "Resolution"
        ] == 0.4
    ]
    .round(2)
)


print("\n[7] Resolution 0.6 Cluster组成")
print("-" * 110)

display(
    cnv_cluster_composition[
        cnv_cluster_composition[
            "Resolution"
        ] == 0.6
    ]
    .round(2)
)


print("\n" + "=" * 110)

print(
    "目前尚未进行 malignant / non-malignant 最终分类。"
)

print(
    "下一步："
)

print(
    "选择CNV结构分辨率 → "
    "正常参考敏感性分析 → "
    "形成多证据恶性判定。"
)

print("=" * 110)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP22C - REVIEW OUTPUT FOR NEXT DECISION
# 一次性输出后续恶性判定所需核心结果
# ============================================================

from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score


# ============================================================
# 0. 路径与对象
# ============================================================

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"


STEP22C_CHECKPOINT = (
    DATA_DIR /
    "GSE149614_STEP22C_CNV_PCA_UMAP_CLUSTERED.h5ad"
)


# 如果当前Notebook内存里没有adata_cnv，自动读取
if "adata_cnv" not in globals():

    assert STEP22C_CHECKPOINT.exists(), (
        f"找不到Step22C checkpoint：\n"
        f"{STEP22C_CHECKPOINT}"
    )

    adata_cnv = sc.read_h5ad(
        STEP22C_CHECKPOINT
    )

    print(
        "已读取：",
        STEP22C_CHECKPOINT
    )


print("=" * 115)
print("STEP22C CNV质量与聚类结果汇总")
print("=" * 115)

print(
    "Cells:",
    f"{adata_cnv.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_cnv.n_vars:,}"
)

print(
    "CNV matrix:",
    adata_cnv.obsm[
        "X_cnv"
    ].shape
)


# ============================================================
# 1. CNV burden × 技术指标
# ============================================================

TECHNICAL_VARS = [
    x
    for x in [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt"
    ]
    if x in adata_cnv.obs.columns
]


confound_rows = []


# ------------------------------------------------------------
# 全体细胞
# ------------------------------------------------------------

for var in TECHNICAL_VARS:

    temp = (
        adata_cnv.obs[
            [
                "cnv_burden_mean_abs",
                var
            ]
        ]
        .dropna()
    )

    rho, p = spearmanr(
        temp[
            "cnv_burden_mean_abs"
        ],
        temp[var]
    )

    confound_rows.append({

        "Scope":
            "All_cells",

        "Group":
            "All",

        "Technical_variable":
            var,

        "N_cells":
            len(temp),

        "Spearman_rho":
            rho,

        "P_value":
            p
    })


# ------------------------------------------------------------
# 各major celltype内部
# ------------------------------------------------------------

for celltype in sorted(
    adata_cnv.obs[
        "major_celltype"
    ]
    .astype(str)
    .unique()
):

    mask = (
        adata_cnv.obs[
            "major_celltype"
        ]
        .astype(str)
        == celltype
    )

    for var in TECHNICAL_VARS:

        temp = (
            adata_cnv.obs.loc[
                mask,
                [
                    "cnv_burden_mean_abs",
                    var
                ]
            ]
            .dropna()
        )

        if len(temp) < 30:
            continue

        rho, p = spearmanr(
            temp[
                "cnv_burden_mean_abs"
            ],
            temp[var]
        )

        confound_rows.append({

            "Scope":
                "Within_celltype",

            "Group":
                celltype,

            "Technical_variable":
                var,

            "N_cells":
                len(temp),

            "Spearman_rho":
                rho,

            "P_value":
                p
        })


cnv_confound = pd.DataFrame(
    confound_rows
)


print("\n[1] 全体细胞 CNV burden × 技术指标")
print("-" * 115)

display(
    cnv_confound[
        cnv_confound[
            "Scope"
        ] == "All_cells"
    ]
    .round(5)
)


print("\n[2] 各细胞类型内部 CNV burden × 技术指标")
print("-" * 115)

display(
    cnv_confound[
        cnv_confound[
            "Scope"
        ] == "Within_celltype"
    ]
    .round(5)
)


# ============================================================
# 2. CNV clustering resolution summary
# ============================================================

cnv_resolutions = [
    0.2,
    0.4,
    0.6,
    0.8
]


cluster_summary_rows = []


for res in cnv_resolutions:

    key = (
        "cnv_leiden_r"
        +
        str(res).replace(
            ".",
            "_"
        )
    )

    assert key in adata_cnv.obs.columns, (
        f"缺失：{key}"
    )

    sizes = (
        adata_cnv.obs[
            key
        ]
        .value_counts()
    )

    cluster_summary_rows.append({

        "Resolution":
            res,

        "N_clusters":
            int(
                len(sizes)
            ),

        "Smallest_cluster":
            int(
                sizes.min()
            ),

        "Largest_cluster":
            int(
                sizes.max()
            ),

        "Median_cluster_size":
            float(
                sizes.median()
            ),

        "Clusters_lt_100":
            int(
                (
                    sizes < 100
                ).sum()
            )
    })


cnv_cluster_summary = pd.DataFrame(
    cluster_summary_rows
)


print("\n[3] CNV不同分辨率聚类结果")
print("-" * 115)

display(
    cnv_cluster_summary
)


# ============================================================
# 3. ARI
# ============================================================

ari_rows = []


for r1, r2 in [
    (0.2, 0.4),
    (0.4, 0.6),
    (0.6, 0.8)
]:

    key1 = (
        "cnv_leiden_r"
        +
        str(r1).replace(
            ".",
            "_"
        )
    )

    key2 = (
        "cnv_leiden_r"
        +
        str(r2).replace(
            ".",
            "_"
        )
    )

    ari = adjusted_rand_score(
        adata_cnv.obs[
            key1
        ].astype(str),

        adata_cnv.obs[
            key2
        ].astype(str)
    )

    ari_rows.append({

        "Resolution_1":
            r1,

        "Resolution_2":
            r2,

        "ARI":
            ari
    })


cnv_resolution_ari = pd.DataFrame(
    ari_rows
)


print("\n[4] CNV聚类稳定性 ARI")
print("-" * 115)

display(
    cnv_resolution_ari.round(4)
)


# ============================================================
# 4. Cluster composition helper
# ============================================================

def build_cluster_composition(
    adata,
    resolution
):

    key = (
        "cnv_leiden_r"
        +
        str(resolution).replace(
            ".",
            "_"
        )
    )

    clusters = sorted(
        adata.obs[
            key
        ]
        .astype(str)
        .unique(),
        key=lambda x:
        int(x)
    )

    rows = []


    for cluster in clusters:

        mask = (
            adata.obs[
                key
            ]
            .astype(str)
            == cluster
        )

        obs_sub = (
            adata.obs.loc[
                mask
            ]
        )

        n = len(
            obs_sub
        )


        def pct(condition):

            return float(
                np.mean(
                    condition
                )
                *
                100
            )


        # ----------------------------------------------------
        # patient dominance
        # ----------------------------------------------------

        patient_counts = (
            obs_sub[
                "patient"
            ]
            .astype(str)
            .value_counts()
        )

        dominant_patient = (
            patient_counts.index[0]
        )

        dominant_patient_pct = (
            patient_counts.iloc[0]
            /
            n
            *
            100
        )


        sample_counts = (
            obs_sub[
                "sample"
            ]
            .astype(str)
            .value_counts()
        )

        dominant_sample = (
            sample_counts.index[0]
        )

        dominant_sample_pct = (
            sample_counts.iloc[0]
            /
            n
            *
            100
        )


        rows.append({

            "Resolution":
                resolution,

            "CNV_cluster":
                cluster,

            "N_cells":
                n,

            "Median_CNV":
                float(
                    obs_sub[
                        "cnv_burden_mean_abs"
                    ].median()
                ),

            "Mean_CNV":
                float(
                    obs_sub[
                        "cnv_burden_mean_abs"
                    ].mean()
                ),

            "P95_CNV":
                float(
                    obs_sub[
                        "cnv_burden_mean_abs"
                    ].quantile(
                        0.95
                    )
                ),

            # -----------------------------------------------
            # reference
            # -----------------------------------------------

            "Pct_Normal_Hep_reference":
                pct(
                    obs_sub[
                        "cnv_reference_class"
                    ]
                    .astype(str)
                    ==
                    "Normal_Hepatocyte"
                ),

            "Pct_Normal_Ductal_reference":
                pct(
                    obs_sub[
                        "cnv_reference_class"
                    ]
                    .astype(str)
                    ==
                    "Normal_Ductal"
                ),

            # -----------------------------------------------
            # major celltype
            # -----------------------------------------------

            "Pct_Atypical_Epithelial":
                pct(
                    obs_sub[
                        "major_celltype"
                    ]
                    .astype(str)
                    ==
                    "Atypical_Epithelial"
                ),

            "Pct_Hepatocyte":
                pct(
                    obs_sub[
                        "major_celltype"
                    ]
                    .astype(str)
                    ==
                    "Hepatocyte"
                ),

            "Pct_Ductal_Epithelial":
                pct(
                    obs_sub[
                        "major_celltype"
                    ]
                    .astype(str)
                    ==
                    "Ductal_Epithelial"
                ),

            # -----------------------------------------------
            # site
            # -----------------------------------------------

            "Pct_Normal_site":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "Normal"
                ),

            "Pct_Tumor_site":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "Tumor"
                ),

            "Pct_PVTT":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "PVTT"
                ),

            "Pct_Lymph":
                pct(
                    obs_sub[
                        "site"
                    ]
                    .astype(str)
                    ==
                    "Lymph"
                ),

            # -----------------------------------------------
            # patient/sample dominance
            # -----------------------------------------------

            "Dominant_patient":
                dominant_patient,

            "Dominant_patient_pct":
                dominant_patient_pct,

            "Dominant_sample":
                dominant_sample,

            "Dominant_sample_pct":
                dominant_sample_pct
        })


    return pd.DataFrame(
        rows
    )


# ============================================================
# 5. Resolution 0.4
# ============================================================

cnv_cluster_r04 = (
    build_cluster_composition(
        adata_cnv,
        0.4
    )
)


print("\n[5] Resolution 0.4 Cluster组成")
print("-" * 115)

display(
    cnv_cluster_r04.round(3)
)


# ============================================================
# 6. Resolution 0.6
# ============================================================

cnv_cluster_r06 = (
    build_cluster_composition(
        adata_cnv,
        0.6
    )
)


print("\n[6] Resolution 0.6 Cluster组成")
print("-" * 115)

display(
    cnv_cluster_r06.round(3)
)


# ============================================================
# 7. 额外输出：
# 正常参考落在哪些CNV cluster
# ============================================================

print("\n[7] 正常参考细胞的CNV Cluster分布")
print("-" * 115)


for res in [
    0.4,
    0.6
]:

    key = (
        "cnv_leiden_r"
        +
        str(res).replace(
            ".",
            "_"
        )
    )

    ref_mask = (
        adata_cnv.obs[
            "cnv_reference_class"
        ]
        .astype(str)
        .isin(
            [
                "Normal_Hepatocyte",
                "Normal_Ductal"
            ]
        )
    )

    ref_tab = pd.crosstab(
        adata_cnv.obs.loc[
            ref_mask,
            key
        ],
        adata_cnv.obs.loc[
            ref_mask,
            "cnv_reference_class"
        ]
    )

    print(
        f"\nResolution = {res}"
    )

    display(
        ref_tab
    )


# ============================================================
# 8. 图像检查
# ============================================================

print("\n[8] 需要截图给我的CNV UMAP")
print("-" * 115)

figure_names = [

    "Step22C_04_CNV_UMAP_ByReference",

    "Step22C_05_CNV_UMAP_ByCelltype",

    "Step22C_07_CNV_UMAP_ByPatient"
]


for name in figure_names:

    png_path = (
        FIGURES_DIR /
        f"{name}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{name}.pdf"
    )

    print(
        name,
        "| PNG:",
        "OK"
        if png_path.exists()
        else "MISSING",
        "| PDF:",
        "OK"
        if pdf_path.exists()
        else "MISSING"
    )


# ============================================================
# 9. 全部保存
# ============================================================

cnv_confound.to_csv(
    RESULTS_DIR /
    "Step22C_REVIEW_TechnicalConfounding.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_cluster_summary.to_csv(
    RESULTS_DIR /
    "Step22C_REVIEW_ResolutionSummary.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_resolution_ari.to_csv(
    RESULTS_DIR /
    "Step22C_REVIEW_ResolutionARI.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_cluster_r04.to_csv(
    RESULTS_DIR /
    "Step22C_REVIEW_ClusterComposition_r0_4.csv",
    index=False,
    encoding="utf-8-sig"
)

cnv_cluster_r06.to_csv(
    RESULTS_DIR /
    "Step22C_REVIEW_ClusterComposition_r0_6.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 115)
print("STEP22C REVIEW OUTPUT COMPLETE")
print("=" * 115)

print(
    "\n请把上面的 [1]、[3]、[4]、[5]、[6] "
    "结果给我。"
)

print(
    "另外发送：ByReference、ByCelltype、ByPatient "
    "三张CNV UMAP。"
)

print(
    "\n下一步不会直接设CNV阈值，"
    "而是先确定CNV结构、正常参考分布以及患者特异性，"
    "然后再进行患者平衡参考敏感性分析。"
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP22C-2 - CELL 45
# LOAD PRIMARY CNV RESULT
# ============================================================

from pathlib import Path
import scanpy as sc
import anndata as ad
import infercnvpy as cnv
import pandas as pd
import numpy as np
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import adjusted_rand_score
import gc
import time

SEED = 42
np.random.seed(SEED)

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

INPUT = (
    DATA_DIR /
    "GSE149614_STEP22C_CNV_PCA_UMAP_CLUSTERED.h5ad"
)

assert INPUT.exists()

adata_sens = sc.read_h5ad(
    INPUT
)

print("=" * 100)
print("STEP22C-2 SENSITIVITY ANALYSIS")
print("=" * 100)

print(
    "Shape:",
    adata_sens.shape
)

print(
    "Primary CNV:",
    adata_sens.obsm["X_cnv"].shape
)

print(
    "Primary r0.4 clusters:",
    adata_sens.obs[
        "cnv_leiden_r0_4"
    ].nunique()
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 46
# PATIENT-CAPPED NORMAL REFERENCE
# ============================================================

HEP_CAP = 150
DUCTAL_CAP = 50


hep_ref_mask = (

    (
        adata_sens.obs["major_celltype"]
        .astype(str)
        == "Hepatocyte"
    )

    &

    (
        adata_sens.obs["site"]
        .astype(str)
        == "Normal"
    )
)


ductal_ref_mask = (

    (
        adata_sens.obs["major_celltype"]
        .astype(str)
        == "Ductal_Epithelial"
    )

    &

    (
        adata_sens.obs["site"]
        .astype(str)
        == "Normal"
    )
)


def patient_capped_sample(
    obs,
    mask,
    cap,
    seed
):

    selected = []

    patients = sorted(
        obs.loc[
            mask,
            "patient"
        ]
        .astype(str)
        .unique()
    )

    for i, patient in enumerate(
        patients
    ):

        idx = (
            obs.index[
                mask
                &
                (
                    obs["patient"]
                    .astype(str)
                    == patient
                )
            ]
            .tolist()
        )

        n_take = min(
            len(idx),
            cap
        )

        rng = np.random.default_rng(
            seed + i
        )

        chosen = rng.choice(
            idx,
            size=n_take,
            replace=False
        )

        selected.extend(
            chosen.tolist()
        )

    return selected


balanced_hep_cells = (
    patient_capped_sample(
        adata_sens.obs,
        hep_ref_mask,
        HEP_CAP,
        42
    )
)


balanced_ductal_cells = (
    patient_capped_sample(
        adata_sens.obs,
        ductal_ref_mask,
        DUCTAL_CAP,
        420
    )
)


adata_sens.obs[
    "cnv_reference_balanced"
] = "Query"


adata_sens.obs.loc[
    balanced_hep_cells,
    "cnv_reference_balanced"
] = "Normal_Hepatocyte"


adata_sens.obs.loc[
    balanced_ductal_cells,
    "cnv_reference_balanced"
] = "Normal_Ductal"


adata_sens.obs[
    "cnv_reference_balanced"
] = (
    adata_sens.obs[
        "cnv_reference_balanced"
    ]
    .astype("category")
)


print("=" * 100)
print("PATIENT-CAPPED REFERENCE")
print("=" * 100)

display(
    pd.crosstab(
        adata_sens.obs[
            "patient"
        ],
        adata_sens.obs[
            "cnv_reference_balanced"
        ]
    )
)

print(
    "\nBalanced hepatocytes:",
    len(
        balanced_hep_cells
    )
)

print(
    "Balanced ductal:",
    len(
        balanced_ductal_cells
    )
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 47
# SENSITIVITY A
# PATIENT-BALANCED REFERENCE
# ============================================================

print("=" * 100)
print("SENSITIVITY A - BALANCED REFERENCE")
print("=" * 100)

start_time = time.time()


cnv.tl.infercnv(

    adata_sens,

    reference_key=
        "cnv_reference_balanced",

    reference_cat=[
        "Normal_Hepatocyte",
        "Normal_Ductal"
    ],

    lfc_clip=3,

    window_size=100,

    step=10,

    dynamic_threshold=1.5,

    exclude_chromosomes=(
        "chrX",
        "chrY"
    ),

    chunksize=1000,

    n_jobs=2,

    inplace=True,

    key_added=
        "cnv_balanced",

    calculate_gene_values=False
)


print(
    "Elapsed minutes:",
    round(
        (
            time.time()
            -
            start_time
        )
        / 60,
        2
    )
)

print(
    "Balanced CNV matrix:",
    adata_sens.obsm[
        "X_cnv_balanced"
    ].shape
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 48
# SENSITIVITY B
# NORMAL HEPATOCYTE ONLY
# ============================================================

adata_sens.obs[
    "cnv_reference_heponly"
] = "Query"


adata_sens.obs.loc[
    hep_ref_mask,
    "cnv_reference_heponly"
] = "Normal_Hepatocyte"


adata_sens.obs[
    "cnv_reference_heponly"
] = (
    adata_sens.obs[
        "cnv_reference_heponly"
    ]
    .astype("category")
)


print("=" * 100)
print("SENSITIVITY B - HEPATOCYTE ONLY")
print("=" * 100)

print(
    "Normal hepatocyte reference:",
    int(
        hep_ref_mask.sum()
    )
)


start_time = time.time()


cnv.tl.infercnv(

    adata_sens,

    reference_key=
        "cnv_reference_heponly",

    reference_cat=[
        "Normal_Hepatocyte"
    ],

    lfc_clip=3,

    window_size=100,

    step=10,

    dynamic_threshold=1.5,

    exclude_chromosomes=(
        "chrX",
        "chrY"
    ),

    chunksize=1000,

    n_jobs=2,

    inplace=True,

    key_added=
        "cnv_heponly",

    calculate_gene_values=False
)


print(
    "Elapsed minutes:",
    round(
        (
            time.time()
            -
            start_time
        )
        / 60,
        2
    )
)

print(
    "Hep-only CNV matrix:",
    adata_sens.obsm[
        "X_cnv_heponly"
    ].shape
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 49
# CALCULATE CNV BURDEN FOR ALL SCENARIOS
# ============================================================

def calculate_mean_abs(
    X
):

    if sparse.issparse(X):

        return (
            np.asarray(
                abs(X)
                .mean(axis=1)
            )
            .ravel()
        )

    X = np.asarray(X)

    return np.mean(
        np.abs(X),
        axis=1
    )


adata_sens.obs[
    "cnv_burden_primary"
] = calculate_mean_abs(
    adata_sens.obsm[
        "X_cnv"
    ]
)


adata_sens.obs[
    "cnv_burden_balanced"
] = calculate_mean_abs(
    adata_sens.obsm[
        "X_cnv_balanced"
    ]
)


adata_sens.obs[
    "cnv_burden_heponly"
] = calculate_mean_abs(
    adata_sens.obsm[
        "X_cnv_heponly"
    ]
)


print("=" * 100)
print("CNV BURDEN CALCULATED")
print("=" * 100)

display(
    adata_sens.obs[
        [
            "cnv_burden_primary",
            "cnv_burden_balanced",
            "cnv_burden_heponly"
        ]
    ]
    .describe()
    .T
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 50
# BURDEN SENSITIVITY CORRELATION
# ============================================================

comparison_pairs = [

    (
        "cnv_burden_primary",
        "cnv_burden_balanced"
    ),

    (
        "cnv_burden_primary",
        "cnv_burden_heponly"
    ),

    (
        "cnv_burden_balanced",
        "cnv_burden_heponly"
    )
]


correlation_rows = []


for x, y in comparison_pairs:

    rho, p = spearmanr(
        adata_sens.obs[x],
        adata_sens.obs[y]
    )

    correlation_rows.append({

        "Metric_1":
            x,

        "Metric_2":
            y,

        "Spearman_rho":
            rho,

        "P_value":
            p
    })


cnv_sensitivity_correlations = (
    pd.DataFrame(
        correlation_rows
    )
)


print("=" * 100)
print("CNV SENSITIVITY CORRELATION")
print("=" * 100)

display(
    cnv_sensitivity_correlations
    .round(5)
)


cnv_sensitivity_correlations.to_csv(
    RESULTS_DIR /
    "Step22C_Sensitivity_CNV_Burden_Correlation.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 51
# CNV BURDEN BY CELLTYPE ACROSS SCENARIOS
# ============================================================

sensitivity_by_celltype = (

    adata_sens.obs
    .groupby(
        "major_celltype",
        observed=True
    )
    .agg(

        N_cells=(
            "cnv_burden_primary",
            "size"
        ),

        Primary_median=(
            "cnv_burden_primary",
            "median"
        ),

        Balanced_median=(
            "cnv_burden_balanced",
            "median"
        ),

        HepOnly_median=(
            "cnv_burden_heponly",
            "median"
        ),

        Primary_mean=(
            "cnv_burden_primary",
            "mean"
        ),

        Balanced_mean=(
            "cnv_burden_balanced",
            "mean"
        ),

        HepOnly_mean=(
            "cnv_burden_heponly",
            "mean"
        )
    )
    .reset_index()
)


print("=" * 100)
print("SENSITIVITY BY CELLTYPE")
print("=" * 100)

display(
    sensitivity_by_celltype
    .round(5)
)


sensitivity_by_celltype.to_csv(
    RESULTS_DIR /
    "Step22C_Sensitivity_ByCelltype.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 52
# CLUSTER SENSITIVITY AT RESOLUTION 0.4
# ============================================================

def cluster_cnv_representation(
    X,
    obs,
    prefix
):

    temp = ad.AnnData(
        X=(
            X.copy()
            if sparse.issparse(X)
            else np.asarray(
                X,
                dtype=np.float32
            )
        ),

        obs=
            obs.copy()
    )


    sc.tl.pca(
        temp,
        n_comps=30,
        svd_solver="arpack",
        random_state=42
    )


    sc.pp.neighbors(
        temp,
        n_neighbors=15,
        n_pcs=20,
        use_rep="X_pca",
        random_state=42
    )


    sc.tl.leiden(
        temp,
        resolution=0.4,
        key_added="leiden",
        random_state=42,
        flavor="igraph",
        n_iterations=2,
        directed=False
    )


    return (
        temp.obs[
            "leiden"
        ]
        .astype(str)
        .values
    )


print(
    "Clustering balanced-reference CNV..."
)

adata_sens.obs[
    "cnv_balanced_leiden_r0_4"
] = cluster_cnv_representation(

    adata_sens.obsm[
        "X_cnv_balanced"
    ],

    adata_sens.obs,

    "balanced"
)


print(
    "Clustering hepatocyte-only CNV..."
)

adata_sens.obs[
    "cnv_heponly_leiden_r0_4"
] = cluster_cnv_representation(

    adata_sens.obsm[
        "X_cnv_heponly"
    ],

    adata_sens.obs,

    "heponly"
)


print(
    "\nBalanced clusters:",
    adata_sens.obs[
        "cnv_balanced_leiden_r0_4"
    ].nunique()
)

print(
    "Hep-only clusters:",
    adata_sens.obs[
        "cnv_heponly_leiden_r0_4"
    ].nunique()
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 53
# CLUSTER STABILITY
# ============================================================

primary_cluster = (
    adata_sens.obs[
        "cnv_leiden_r0_4"
    ]
    .astype(str)
)


balanced_cluster = (
    adata_sens.obs[
        "cnv_balanced_leiden_r0_4"
    ]
    .astype(str)
)


heponly_cluster = (
    adata_sens.obs[
        "cnv_heponly_leiden_r0_4"
    ]
    .astype(str)
)


ari_primary_balanced = (
    adjusted_rand_score(
        primary_cluster,
        balanced_cluster
    )
)


ari_primary_heponly = (
    adjusted_rand_score(
        primary_cluster,
        heponly_cluster
    )
)


ari_balanced_heponly = (
    adjusted_rand_score(
        balanced_cluster,
        heponly_cluster
    )
)


cluster_sensitivity = pd.DataFrame({

    "Comparison": [
        "Primary_vs_Balanced",
        "Primary_vs_HepOnly",
        "Balanced_vs_HepOnly"
    ],

    "ARI": [
        ari_primary_balanced,
        ari_primary_heponly,
        ari_balanced_heponly
    ]
})


print("=" * 100)
print("CNV CLUSTER SENSITIVITY")
print("=" * 100)

display(
    cluster_sensitivity
    .round(4)
)


cluster_sensitivity.to_csv(
    RESULTS_DIR /
    "Step22C_Sensitivity_CNV_Cluster_ARI.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 54
# PRIMARY CNV CLUSTER ROBUSTNESS
# ============================================================

cluster_burden_sensitivity = (

    adata_sens.obs
    .groupby(
        "cnv_leiden_r0_4",
        observed=True
    )
    .agg(

        N_cells=(
            "cnv_burden_primary",
            "size"
        ),

        Primary_median=(
            "cnv_burden_primary",
            "median"
        ),

        Balanced_median=(
            "cnv_burden_balanced",
            "median"
        ),

        HepOnly_median=(
            "cnv_burden_heponly",
            "median"
        ),

        Primary_mean=(
            "cnv_burden_primary",
            "mean"
        ),

        Balanced_mean=(
            "cnv_burden_balanced",
            "mean"
        ),

        HepOnly_mean=(
            "cnv_burden_heponly",
            "mean"
        )
    )
    .reset_index()
)


cluster_burden_sensitivity[
    "cnv_leiden_r0_4"
] = (
    cluster_burden_sensitivity[
        "cnv_leiden_r0_4"
    ]
    .astype(str)
)


cluster_burden_sensitivity = (
    cluster_burden_sensitivity
    .sort_values(
        "cnv_leiden_r0_4",
        key=lambda x:
        x.astype(int)
    )
)


print("=" * 100)
print("PRIMARY r0.4 CLUSTER ROBUSTNESS")
print("=" * 100)

display(
    cluster_burden_sensitivity
    .round(5)
)


cluster_burden_sensitivity.to_csv(
    RESULTS_DIR /
    "Step22C_Sensitivity_PrimaryCluster_Burden.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 55
# SAVE SENSITIVITY CHECKPOINT
# ============================================================

SENSITIVITY_CHECKPOINT = (
    DATA_DIR /
    "GSE149614_STEP22C_CNV_REFERENCE_SENSITIVITY.h5ad"
)


adata_sens.write_h5ad(
    SENSITIVITY_CHECKPOINT,
    compression="gzip"
)


print("=" * 100)
print("SENSITIVITY CHECKPOINT SAVED")
print("=" * 100)

print(
    SENSITIVITY_CHECKPOINT
)

In [ ]:
# ============================================================
# STEP22C-2 - CELL 56
# FINAL SENSITIVITY REVIEW
# ============================================================

print("=" * 110)
print("STEP22C-2 CNV参考敏感性分析")
print("=" * 110)


print("\n[1] CNV burden相关性")
print("-" * 110)

display(
    cnv_sensitivity_correlations
    .round(5)
)


print("\n[2] 三类细胞在三套参考下的CNV")
print("-" * 110)

display(
    sensitivity_by_celltype
    .round(5)
)


print("\n[3] CNV聚类稳定性")
print("-" * 110)

display(
    cluster_sensitivity
    .round(4)
)


print("\n[4] Primary r0.4各Cluster稳定性")
print("-" * 110)

display(
    cluster_burden_sensitivity
    .round(5)
)


print("\n" + "=" * 110)

print(
    "Sensitivity analysis complete."
)

print(
    "Malignant calling has NOT yet been performed."
)

print("=" * 110)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP22D - CELL 57
# 初始化最终恶性细胞判定
# ============================================================

from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import sparse
from scipy.stats import (
    spearmanr,
    mannwhitneyu
)
import infercnvpy as cnv
import json
import gc


SEED = 42
np.random.seed(SEED)


ROOT = Path(
    _hcc_path('LIHC_scRNA_Step22_Malignant_Identification')
)

DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"


STEP22C_INPUT = (
    DATA_DIR /
    "GSE149614_STEP22C_CNV_REFERENCE_SENSITIVITY.h5ad"
)


assert STEP22C_INPUT.exists(), (
    f"找不到：\n{STEP22C_INPUT}"
)


adata_final = sc.read_h5ad(
    STEP22C_INPUT
)


CNV_CLUSTER_KEY = (
    "cnv_leiden_r0_4"
)


required_obs = [

    "major_celltype",
    "site",
    "sample",
    "patient",
    "cnv_reference_class",
    CNV_CLUSTER_KEY

]


for col in required_obs:

    assert col in adata_final.obs.columns, (
        f"缺少obs字段：{col}"
    )


assert "X_cnv" in adata_final.obsm


print("=" * 100)
print("STEP22D FINAL MALIGNANCY CALLING")
print("=" * 100)

print(
    "Cells:",
    f"{adata_final.n_obs:,}"
)

print(
    "Genes:",
    f"{adata_final.n_vars:,}"
)

print(
    "CNV representation:",
    adata_final.obsm[
        "X_cnv"
    ].shape
)

print(
    "Primary CNV clusters:",
    adata_final.obs[
        CNV_CLUSTER_KEY
    ].nunique()
)

In [ ]:
# ============================================================
# STEP22D - CELL 58
# 生物学表达程序评分
# ============================================================

EXPRESSION_PROGRAMS = {

    "mature_hepatocyte_score": [
        "ALB",
        "APOA1",
        "APOA2",
        "TTR",
        "ASGR1",
        "HPD",
        "CYP2E1",
        "CPS1",
        "MAT1A",
        "ARG1"
    ],

    "ductal_progenitor_score": [
        "EPCAM",
        "KRT19",
        "KRT7",
        "KRT8",
        "KRT18",
        "SOX9",
        "CFTR",
        "TACSTD2"
    ],

    "tumor_associated_expression_score": [
        "AFP",
        "GPC3",
        "DLK1",
        "IGF2",
        "CLDN6",
        "PRAME",
        "SPP1",
        "PAGE1",
        "PAGE2",
        "GAGE12H",
        "GAGE13",
        "SSX3",
        "S100P"
    ]
}


program_rows = []


for score_name, genes in EXPRESSION_PROGRAMS.items():

    detected = [
        g
        for g in genes
        if g in adata_final.var_names
    ]

    missing = [
        g
        for g in genes
        if g not in adata_final.var_names
    ]


    print(
        f"\n{score_name}"
    )

    print(
        "Detected:",
        detected
    )

    print(
        "Missing:",
        missing
    )


    assert len(detected) >= 3


    sc.tl.score_genes(

        adata_final,

        gene_list=
            detected,

        score_name=
            score_name,

        ctrl_size=
            min(
                50,
                max(
                    10,
                    len(detected) * 5
                )
            ),

        n_bins=25,

        random_state=42,

        use_raw=False
    )


    for gene in genes:

        program_rows.append({

            "Score":
                score_name,

            "Gene":
                gene,

            "Detected":
                gene in adata_final.var_names
        })


program_audit = pd.DataFrame(
    program_rows
)


program_audit.to_csv(
    RESULTS_DIR /
    "Step22D_ExpressionProgram_GeneAudit.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\nExpression program scoring = COMPLETE")

In [ ]:
# ============================================================
# STEP22D - CELL 59
# CNV CLUSTER EVIDENCE TABLE
# ============================================================

cluster_rows = []


clusters = sorted(

    adata_final.obs[
        CNV_CLUSTER_KEY
    ]
    .astype(str)
    .unique(),

    key=lambda x:
        int(x)
)


for cluster in clusters:

    mask = (
        adata_final.obs[
            CNV_CLUSTER_KEY
        ]
        .astype(str)
        == cluster
    )


    temp = (
        adata_final.obs.loc[
            mask
        ]
    )


    n = len(
        temp
    )


    def pct(condition):

        return float(
            np.mean(
                condition
            )
            *
            100
        )


    patient_counts = (
        temp[
            "patient"
        ]
        .astype(str)
        .value_counts()
    )


    sample_counts = (
        temp[
            "sample"
        ]
        .astype(str)
        .value_counts()
    )


    normal_hep_pct = pct(

        temp[
            "cnv_reference_class"
        ].astype(str)
        ==
        "Normal_Hepatocyte"

    )


    normal_ductal_pct = pct(

        temp[
            "cnv_reference_class"
        ].astype(str)
        ==
        "Normal_Ductal"

    )


    normal_reference_pct = (
        normal_hep_pct
        +
        normal_ductal_pct
    )


    atypical_pct = pct(

        temp[
            "major_celltype"
        ].astype(str)
        ==
        "Atypical_Epithelial"

    )


    hep_pct = pct(

        temp[
            "major_celltype"
        ].astype(str)
        ==
        "Hepatocyte"

    )


    ductal_pct = pct(

        temp[
            "major_celltype"
        ].astype(str)
        ==
        "Ductal_Epithelial"

    )


    normal_site_pct = pct(

        temp[
            "site"
        ].astype(str)
        ==
        "Normal"

    )


    tumor_site_pct = pct(

        temp[
            "site"
        ].astype(str)
        ==
        "Tumor"

    )


    pvtt_pct = pct(

        temp[
            "site"
        ].astype(str)
        ==
        "PVTT"

    )


    lymph_pct = pct(

        temp[
            "site"
        ].astype(str)
        ==
        "Lymph"

    )


    tumor_associated_pct = (
        tumor_site_pct
        +
        pvtt_pct
        +
        lymph_pct
    )


    # --------------------------------------------------------
    # Structural class
    # --------------------------------------------------------

    if (

        atypical_pct >= 95

        and

        tumor_associated_pct >= 95

        and

        normal_reference_pct <= 1

    ):

        structural_class = (
            "Strong_malignant_candidate"
        )


    elif (

        normal_reference_pct >= 70

        and

        normal_site_pct >= 70

    ):

        structural_class = (
            "Normal_like"
        )


    elif (

        hep_pct >= 95

        and

        tumor_site_pct >= 95

        and

        normal_reference_pct <= 1

    ):

        structural_class = (
            "Tumor_hepatocyte_candidate"
        )


    else:

        structural_class = (
            "Mixed_uncertain"
        )


    cluster_rows.append({

        "CNV_cluster":
            cluster,

        "N_cells":
            n,

        "Median_CNV":
            float(
                temp[
                    "cnv_burden_primary"
                ].median()
            ),

        "Mean_CNV":
            float(
                temp[
                    "cnv_burden_primary"
                ].mean()
            ),

        "Pct_Normal_reference":
            normal_reference_pct,

        "Pct_Atypical_Epithelial":
            atypical_pct,

        "Pct_Hepatocyte":
            hep_pct,

        "Pct_Ductal_Epithelial":
            ductal_pct,

        "Pct_Normal_site":
            normal_site_pct,

        "Pct_Tumor":
            tumor_site_pct,

        "Pct_PVTT":
            pvtt_pct,

        "Pct_Lymph":
            lymph_pct,

        "Pct_Tumor_associated":
            tumor_associated_pct,

        "Dominant_patient":
            patient_counts.index[0],

        "Dominant_patient_pct":
            (
                patient_counts.iloc[0]
                /
                n
                *
                100
            ),

        "Dominant_sample":
            sample_counts.index[0],

        "Dominant_sample_pct":
            (
                sample_counts.iloc[0]
                /
                n
                *
                100
            ),

        "Median_MatureHep_score":
            float(
                temp[
                    "mature_hepatocyte_score"
                ].median()
            ),

        "Median_Ductal_score":
            float(
                temp[
                    "ductal_progenitor_score"
                ].median()
            ),

        "Median_TumorExpression_score":
            float(
                temp[
                    "tumor_associated_expression_score"
                ].median()
            ),

        "Structural_class":
            structural_class
    })


cluster_evidence = pd.DataFrame(
    cluster_rows
)


cluster_evidence = (
    cluster_evidence
    .sort_values(
        "CNV_cluster",
        key=lambda x:
            x.astype(int)
    )
    .reset_index(drop=True)
)


print("=" * 115)
print("CNV CLUSTER EVIDENCE")
print("=" * 115)

display(
    cluster_evidence.round(3)
)


cluster_evidence.to_csv(
    RESULTS_DIR /
    "Step22D_CNV_Cluster_Evidence.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22D - CELL 60
# PAIRED / POOLED NORMAL CNV DISTANCE
# ============================================================

X_CNV = (
    adata_final.obsm[
        "X_cnv"
    ]
)


def get_dense_rows(
    X,
    row_indices
):

    temp = X[
        row_indices
    ]

    if sparse.issparse(
        temp
    ):

        return (
            temp
            .toarray()
            .astype(
                np.float32
            )
        )

    return np.asarray(
        temp,
        dtype=np.float32
    )


def reference_distance_distribution(
    reference_matrix
):
    """
    Leave-one-out reference distance.
    """

    n = (
        reference_matrix.shape[0]
    )


    if n < 2:

        raise ValueError(
            "Reference cells < 2"
        )


    total_sum = (
        reference_matrix
        .sum(
            axis=0
        )
    )


    loo_centroid = (

        total_sum[
            None,
            :
        ]

        -

        reference_matrix

    ) / (
        n - 1
    )


    distances = np.mean(

        np.abs(

            reference_matrix
            -
            loo_centroid

        ),

        axis=1
    )


    return distances


def query_distance_to_centroid(
    query_matrix,
    centroid
):

    return np.mean(

        np.abs(

            query_matrix
            -
            centroid[
                None,
                :
            ]

        ),

        axis=1
    )


# ------------------------------------------------------------
# Pooled normal hepatocyte
# ------------------------------------------------------------

pooled_normal_mask = (

    (
        adata_final.obs[
            "major_celltype"
        ]
        .astype(str)
        == "Hepatocyte"
    )

    &

    (
        adata_final.obs[
            "site"
        ]
        .astype(str)
        == "Normal"
    )
)


pooled_normal_indices = np.where(
    pooled_normal_mask.values
)[0]


# ------------------------------------------------------------
# Candidate clusters
# ------------------------------------------------------------

candidate_clusters = (
    cluster_evidence.loc[
        cluster_evidence[
            "Structural_class"
        ]
        ==
        "Tumor_hepatocyte_candidate",
        "CNV_cluster"
    ]
    .astype(str)
    .tolist()
)


print(
    "Tumor-hepatocyte candidate clusters:",
    candidate_clusters
)


paired_rows = []


for cluster in candidate_clusters:

    cluster_mask = (

        adata_final.obs[
            CNV_CLUSTER_KEY
        ]
        .astype(str)
        == cluster
    )


    query_indices = np.where(
        cluster_mask.values
    )[0]


    cluster_obs = (
        adata_final.obs.iloc[
            query_indices
        ]
    )


    dominant_patient = (

        cluster_obs[
            "patient"
        ]
        .astype(str)
        .value_counts()
        .index[0]
    )


    # --------------------------------------------------------
    # Same-patient normal hepatocyte
    # --------------------------------------------------------

    paired_mask = (

        pooled_normal_mask

        &

        (
            adata_final.obs[
                "patient"
            ]
            .astype(str)
            ==
            dominant_patient
        )
    )


    paired_indices = np.where(
        paired_mask.values
    )[0]


    n_paired = len(
        paired_indices
    )


    # --------------------------------------------------------
    # ≥20 cells → patient-paired normal
    # otherwise → pooled normal
    # --------------------------------------------------------

    if n_paired >= 20:

        formal_reference_indices = (
            paired_indices
        )

        reference_mode = (
            "Patient_paired_normal"
        )

    else:

        formal_reference_indices = (
            pooled_normal_indices
        )

        reference_mode = (
            "Pooled_normal_hepatocyte"
        )


    R = get_dense_rows(
        X_CNV,
        formal_reference_indices
    )


    Q = get_dense_rows(
        X_CNV,
        query_indices
    )


    reference_centroid = (
        R.mean(
            axis=0
        )
    )


    ref_dist = (
        reference_distance_distribution(
            R
        )
    )


    query_dist = (
        query_distance_to_centroid(
            Q,
            reference_centroid
        )
    )


    ref_median = float(
        np.median(
            ref_dist
        )
    )

    ref_p95 = float(
        np.quantile(
            ref_dist,
            0.95
        )
    )

    ref_p99 = float(
        np.quantile(
            ref_dist,
            0.99
        )
    )


    query_median = float(
        np.median(
            query_dist
        )
    )


    query_centroid = (
        Q.mean(
            axis=0
        )
    )


    centroid_distance = float(
        np.mean(
            np.abs(
                query_centroid
                -
                reference_centroid
            )
        )
    )


    pct_above_p95 = float(
        np.mean(
            query_dist
            >
            ref_p95
        )
        *
        100
    )


    median_ratio = float(
        query_median
        /
        ref_median
    )


    # --------------------------------------------------------
    # Statistical comparison
    # --------------------------------------------------------

    u_stat, p_value = (
        mannwhitneyu(

            query_dist,
            ref_dist,

            alternative=
                "greater"
        )
    )


    rank_biserial = (

        2
        *
        u_stat

        /
        (
            len(query_dist)
            *
            len(ref_dist)
        )

        -
        1
    )


    paired_rows.append({

        "CNV_cluster":
            cluster,

        "Dominant_patient":
            dominant_patient,

        "N_query":
            len(
                query_indices
            ),

        "N_same_patient_normal":
            n_paired,

        "Reference_mode":
            reference_mode,

        "N_reference_used":
            len(
                formal_reference_indices
            ),

        "Reference_distance_median":
            ref_median,

        "Reference_distance_P95":
            ref_p95,

        "Reference_distance_P99":
            ref_p99,

        "Query_distance_median":
            query_median,

        "Query_vs_reference_median_ratio":
            median_ratio,

        "Query_pct_above_reference_P95":
            pct_above_p95,

        "ClusterCentroid_to_reference_distance":
            centroid_distance,

        "MannWhitney_P":
            p_value,

        "Rank_biserial":
            rank_biserial
    })


paired_cnv_evidence = pd.DataFrame(
    paired_rows
)


print("=" * 115)
print("PAIRED NORMAL CNV EVIDENCE")
print("=" * 115)

display(
    paired_cnv_evidence.round(5)
)


paired_cnv_evidence.to_csv(
    RESULTS_DIR /
    "Step22D_TumorHepatocyte_PairedNormal_CNV.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22D - CELL 61
# INTEGRATE CLUSTER-LEVEL EVIDENCE
# ============================================================

# ------------------------------------------------------------
# Conservative decision thresholds
# ------------------------------------------------------------

MALIGNANT_PCT_ABOVE_P95 = 70.0

MALIGNANT_MEDIAN_RATIO = 1.50

NONMALIGNANT_PCT_ABOVE_P95 = 20.0

NONMALIGNANT_MEDIAN_RATIO = 1.20


cluster_final = (
    cluster_evidence
    .copy()
)


cluster_final[
    "Paired_CNV_call"
] = np.nan


cluster_final[
    "Final_cluster_call"
] = "Ambiguous"


cluster_final[
    "Call_reason"
] = ""


# ------------------------------------------------------------
# 1. Strong atypical malignant clone
# ------------------------------------------------------------

mask = (
    cluster_final[
        "Structural_class"
    ]
    ==
    "Strong_malignant_candidate"
)


cluster_final.loc[
    mask,
    "Final_cluster_call"
] = "Malignant"


cluster_final.loc[
    mask,
    "Call_reason"
] = (
    "Atypical epithelial-dominant, "
    "tumor-associated, reference-depleted "
    "patient-specific CNV clone"
)


# ------------------------------------------------------------
# 2. Normal-like CNV cluster
# ------------------------------------------------------------

mask = (
    cluster_final[
        "Structural_class"
    ]
    ==
    "Normal_like"
)


cluster_final.loc[
    mask,
    "Final_cluster_call"
] = (
    "Non_malignant_like"
)


cluster_final.loc[
    mask,
    "Call_reason"
] = (
    "Normal-reference enriched "
    "CNV state"
)


# ------------------------------------------------------------
# 3. Tumor hepatocyte candidates
# ------------------------------------------------------------

for _, row in (
    paired_cnv_evidence
    .iterrows()
):

    cluster = str(
        row[
            "CNV_cluster"
        ]
    )


    idx = (
        cluster_final[
            "CNV_cluster"
        ]
        .astype(str)
        ==
        cluster
    )


    strong_divergence = (

        row[
            "Query_pct_above_reference_P95"
        ]
        >=
        MALIGNANT_PCT_ABOVE_P95

        and

        row[
            "Query_vs_reference_median_ratio"
        ]
        >=
        MALIGNANT_MEDIAN_RATIO

        and

        row[
            "ClusterCentroid_to_reference_distance"
        ]
        >
        row[
            "Reference_distance_P99"
        ]
    )


    normal_like = (

        row[
            "Query_pct_above_reference_P95"
        ]
        <=
        NONMALIGNANT_PCT_ABOVE_P95

        and

        row[
            "Query_vs_reference_median_ratio"
        ]
        <=
        NONMALIGNANT_MEDIAN_RATIO
    )


    if strong_divergence:

        call = (
            "Malignant"
        )

        reason = (
            "Tumor-site mature-hepatocyte cluster "
            "with strong CNV divergence from "
            "patient-matched/pooled normal hepatocytes"
        )


    elif normal_like:

        call = (
            "Non_malignant_like"
        )

        reason = (
            "Tumor-site hepatocyte cluster "
            "with CNV profile close to normal hepatocytes"
        )


    else:

        call = (
            "Ambiguous"
        )

        reason = (
            "Tumor-site hepatocyte cluster "
            "with intermediate/conflicting CNV evidence"
        )


    cluster_final.loc[
        idx,
        "Paired_CNV_call"
    ] = call


    cluster_final.loc[
        idx,
        "Final_cluster_call"
    ] = call


    cluster_final.loc[
        idx,
        "Call_reason"
    ] = reason


print("=" * 115)
print("FINAL CNV CLUSTER CALL")
print("=" * 115)

display(

    cluster_final[
        [
            "CNV_cluster",
            "N_cells",
            "Structural_class",
            "Median_CNV",
            "Pct_Normal_reference",
            "Pct_Atypical_Epithelial",
            "Pct_Hepatocyte",
            "Pct_Normal_site",
            "Pct_Tumor_associated",
            "Dominant_patient",
            "Final_cluster_call",
            "Call_reason"
        ]
    ]
)


cluster_final.to_csv(
    RESULTS_DIR /
    "Step22D_Final_CNV_Cluster_Call.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22D - CELL 62
# FINAL CELL-LEVEL MALIGNANCY STATUS
# ============================================================

cluster_call_map = (
    cluster_final
    .set_index(
        "CNV_cluster"
    )[
        "Final_cluster_call"
    ]
    .to_dict()
)


structural_class_map = (
    cluster_final
    .set_index(
        "CNV_cluster"
    )[
        "Structural_class"
    ]
    .to_dict()
)


adata_final.obs[
    "malignant_status"
] = "Ambiguous"


adata_final.obs[
    "malignancy_confidence"
] = "Low"


adata_final.obs[
    "malignancy_basis"
] = (
    "Insufficient or conflicting evidence"
)


cluster_string = (
    adata_final.obs[
        CNV_CLUSTER_KEY
    ]
    .astype(str)
)


# ============================================================
# 1. Known normal reference cells
# ============================================================

known_reference = (
    adata_final.obs[
        "cnv_reference_class"
    ]
    .astype(str)
    .isin(
        [
            "Normal_Hepatocyte",
            "Normal_Ductal"
        ]
    )
)


adata_final.obs.loc[
    known_reference,
    "malignant_status"
] = "Non_malignant"


adata_final.obs.loc[
    known_reference,
    "malignancy_confidence"
] = "High"


adata_final.obs.loc[
    known_reference,
    "malignancy_basis"
] = (
    "Normal-site reference cell"
)


# ============================================================
# 2. Iterate CNV clusters
# ============================================================

for cluster in clusters:

    cluster_mask = (
        cluster_string
        ==
        cluster
    )


    row = (
        cluster_final.loc[
            cluster_final[
                "CNV_cluster"
            ].astype(str)
            ==
            cluster
        ]
        .iloc[0]
    )


    structural_class = (
        row[
            "Structural_class"
        ]
    )


    cluster_call = (
        row[
            "Final_cluster_call"
        ]
    )


    non_reference_mask = (

        cluster_mask
        &
        (~known_reference)

    )


    # --------------------------------------------------------
    # Strong malignant clone
    # --------------------------------------------------------

    if cluster_call == "Malignant":

        adata_final.obs.loc[
            non_reference_mask,
            "malignant_status"
        ] = "Malignant"


        # Atypical epithelial clone:
        # high confidence
        if (
            structural_class
            ==
            "Strong_malignant_candidate"
        ):

            adata_final.obs.loc[
                non_reference_mask,
                "malignancy_confidence"
            ] = "High"

            adata_final.obs.loc[
                non_reference_mask,
                "malignancy_basis"
            ] = (
                "Atypical epithelial + "
                "tumor-associated origin + "
                "patient-specific abnormal CNV clone"
            )


        # Mature hepatocyte-like malignant candidate:
        # moderate confidence
        else:

            adata_final.obs.loc[
                non_reference_mask,
                "malignancy_confidence"
            ] = "Moderate"

            adata_final.obs.loc[
                non_reference_mask,
                "malignancy_basis"
            ] = (
                "Tumor-site hepatocyte with "
                "CNV divergence from normal reference"
            )


    # --------------------------------------------------------
    # Normal-like CNV cluster
    # --------------------------------------------------------

    elif (

        structural_class
        ==
        "Normal_like"

    ):

        normal_site_nonref = (

            non_reference_mask

            &

            (
                adata_final.obs[
                    "site"
                ]
                .astype(str)
                ==
                "Normal"
            )
        )


        tumor_site_nonref = (

            non_reference_mask
            &
            (~normal_site_nonref)

        )


        adata_final.obs.loc[
            normal_site_nonref,
            "malignant_status"
        ] = "Non_malignant"


        adata_final.obs.loc[
            normal_site_nonref,
            "malignancy_confidence"
        ] = "High"


        adata_final.obs.loc[
            normal_site_nonref,
            "malignancy_basis"
        ] = (
            "Normal-site cell in "
            "normal-reference-enriched CNV state"
        )


        # Tumor-site cells kept ambiguous
        adata_final.obs.loc[
            tumor_site_nonref,
            "malignant_status"
        ] = "Ambiguous"


        adata_final.obs.loc[
            tumor_site_nonref,
            "malignancy_confidence"
        ] = "Low"


        adata_final.obs.loc[
            tumor_site_nonref,
            "malignancy_basis"
        ] = (
            "Tumor-associated site but "
            "normal-like CNV state"
        )


    # --------------------------------------------------------
    # Tumor-hepatocyte CNV looks normal-like
    # --------------------------------------------------------

    elif cluster_call == "Non_malignant_like":

        adata_final.obs.loc[
            non_reference_mask,
            "malignant_status"
        ] = "Non_malignant"


        adata_final.obs.loc[
            non_reference_mask,
            "malignancy_confidence"
        ] = "Moderate"


        adata_final.obs.loc[
            non_reference_mask,
            "malignancy_basis"
        ] = (
            "Tumor-site hepatocyte with "
            "CNV profile close to normal reference"
        )


    # --------------------------------------------------------
    # Ambiguous
    # --------------------------------------------------------

    else:

        adata_final.obs.loc[
            non_reference_mask,
            "malignant_status"
        ] = "Ambiguous"


        adata_final.obs.loc[
            non_reference_mask,
            "malignancy_confidence"
        ] = "Low"


        adata_final.obs.loc[
            non_reference_mask,
            "malignancy_basis"
        ] = (
            "Intermediate or conflicting "
            "malignancy evidence"
        )


# ============================================================
# Category
# ============================================================

status_order = [
    "Non_malignant",
    "Ambiguous",
    "Malignant"
]


adata_final.obs[
    "malignant_status"
] = pd.Categorical(

    adata_final.obs[
        "malignant_status"
    ],

    categories=
        status_order,

    ordered=True
)


print("=" * 100)
print("FINAL CELL-LEVEL MALIGNANCY STATUS")
print("=" * 100)

display(
    adata_final.obs[
        "malignant_status"
    ]
    .value_counts(
        sort=False
    )
    .to_frame(
        "N_cells"
    )
)

In [ ]:
# ============================================================
# STEP22D - CELL 63
# FINAL MALIGNANCY AUDIT
# ============================================================

print("=" * 115)
print("FINAL MALIGNANCY AUDIT")
print("=" * 115)


# ------------------------------------------------------------
# Overall
# ------------------------------------------------------------

final_status_counts = (
    adata_final.obs[
        "malignant_status"
    ]
    .value_counts(
        sort=False
    )
)


final_status_pct = (
    final_status_counts
    /
    adata_final.n_obs
    *
    100
)


final_status_summary = pd.DataFrame({

    "N_cells":
        final_status_counts,

    "Pct_cells":
        final_status_pct
})


print("\n[1] Overall")

display(
    final_status_summary.round(2)
)


# ------------------------------------------------------------
# Status × site
# ------------------------------------------------------------

status_site = pd.crosstab(

    adata_final.obs[
        "malignant_status"
    ],

    adata_final.obs[
        "site"
    ]
)


print("\n[2] Status × Site")

display(
    status_site
)


# ------------------------------------------------------------
# Status × patient
# ------------------------------------------------------------

status_patient = pd.crosstab(

    adata_final.obs[
        "malignant_status"
    ],

    adata_final.obs[
        "patient"
    ]
)


print("\n[3] Status × Patient")

display(
    status_patient
)


# ------------------------------------------------------------
# Status × sample
# ------------------------------------------------------------

status_sample = pd.crosstab(

    adata_final.obs[
        "malignant_status"
    ],

    adata_final.obs[
        "sample"
    ]
)


print("\n[4] Status × Sample")

display(
    status_sample
)


# ------------------------------------------------------------
# Status × major celltype
# ------------------------------------------------------------

status_celltype = pd.crosstab(

    adata_final.obs[
        "malignant_status"
    ],

    adata_final.obs[
        "major_celltype"
    ]
)


print("\n[5] Status × Step21 celltype")

display(
    status_celltype
)


# ------------------------------------------------------------
# Status × CNV cluster
# ------------------------------------------------------------

status_cluster = pd.crosstab(

    adata_final.obs[
        "malignant_status"
    ],

    adata_final.obs[
        CNV_CLUSTER_KEY
    ]
)


print("\n[6] Status × CNV Cluster")

display(
    status_cluster
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

final_status_summary.to_csv(
    RESULTS_DIR /
    "Step22D_Final_Malignancy_Summary.csv",
    encoding="utf-8-sig"
)

status_site.to_csv(
    RESULTS_DIR /
    "Step22D_Malignancy_BySite.csv",
    encoding="utf-8-sig"
)

status_patient.to_csv(
    RESULTS_DIR /
    "Step22D_Malignancy_ByPatient.csv",
    encoding="utf-8-sig"
)

status_sample.to_csv(
    RESULTS_DIR /
    "Step22D_Malignancy_BySample.csv",
    encoding="utf-8-sig"
)

status_celltype.to_csv(
    RESULTS_DIR /
    "Step22D_Malignancy_ByMajorCelltype.csv",
    encoding="utf-8-sig"
)

status_cluster.to_csv(
    RESULTS_DIR /
    "Step22D_Malignancy_ByCNVCluster.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22D - CELL 64
# NORMAL-SITE ATYPICAL EPITHELIAL AUDIT
# ============================================================

normal_atypical_mask = (

    (
        adata_final.obs[
            "major_celltype"
        ]
        .astype(str)
        ==
        "Atypical_Epithelial"
    )

    &

    (
        adata_final.obs[
            "site"
        ]
        .astype(str)
        ==
        "Normal"
    )
)


normal_atypical = (
    adata_final.obs.loc[
        normal_atypical_mask,
        [
            "sample",
            "patient",
            "site",
            CNV_CLUSTER_KEY,
            "cnv_burden_primary",
            "mature_hepatocyte_score",
            "ductal_progenitor_score",
            "tumor_associated_expression_score",
            "malignant_status",
            "malignancy_confidence",
            "malignancy_basis"
        ]
    ]
    .copy()
)


print("=" * 100)
print("NORMAL-SITE ATYPICAL EPITHELIAL")
print("=" * 100)

print(
    "Cells:",
    len(
        normal_atypical
    )
)


print("\nStatus:")

display(
    normal_atypical[
        "malignant_status"
    ]
    .value_counts()
)


print("\nPatient × CNV cluster:")

display(
    pd.crosstab(
        normal_atypical[
            "patient"
        ],
        normal_atypical[
            CNV_CLUSTER_KEY
        ]
    )
)


normal_atypical.to_csv(
    RESULTS_DIR /
    "Step22D_NormalSite_AtypicalEpithelial_Audit.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP22D - CELL 65
# FINAL MALIGNANCY CNV UMAP
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei",
    "SimHei",
    "Arial"
]

plt.rcParams["axes.unicode_minus"] = False


STATUS_COLORS = {

    "Non_malignant":
        "#4DBBD5",

    "Ambiguous":
        "#BDBDBD",

    "Malignant":
        "#E64B35"
}


palette = [
    STATUS_COLORS[x]
    for x in status_order
]


ax = sc.pl.embedding(

    adata_final,

    basis=
        "umap_cnv",

    color=
        "malignant_status",

    palette=
        palette,

    size=7,

    alpha=0.80,

    frameon=False,

    legend_loc=
        "right margin",

    title=
        "CNV空间中的恶性细胞判定",

    show=False
)


fig = ax.figure

fig.set_size_inches(
    8.5,
    6.5
)


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "Step22D_01_Final_Malignancy_CNV_UMAP.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    FIGURES_DIR /
    "Step22D_01_Final_Malignancy_CNV_UMAP.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP22D - CELL 66
# FINAL MALIGNANCY RNA UMAP
# ============================================================

if (
    "X_umap_unintegrated"
    in adata_final.obsm
):

    ax = sc.pl.embedding(

        adata_final,

        basis=
            "umap_unintegrated",

        color=
            "malignant_status",

        palette=
            palette,

        size=7,

        alpha=0.80,

        frameon=False,

        legend_loc=
            "right margin",

        title=
            "RNA表达空间中的恶性细胞判定",

        show=False
    )


    fig = ax.figure

    fig.set_size_inches(
        8.5,
        6.5
    )


    fig.tight_layout()


    fig.savefig(
        FIGURES_DIR /
        "Step22D_02_Final_Malignancy_RNA_UMAP.png",

        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        FIGURES_DIR /
        "Step22D_02_Final_Malignancy_RNA_UMAP.pdf",

        bbox_inches="tight",
        facecolor="white"
    )


    plt.show()
    plt.close(fig)

else:

    print(
        "X_umap_unintegrated 不存在，"
        "跳过RNA UMAP。"
    )

In [ ]:
# ============================================================
# STEP22D - CELL 67
# CNV BURDEN BY FINAL STATUS
# ============================================================

plot_data = []


for status in status_order:

    values = (

        adata_final.obs.loc[

            adata_final.obs[
                "malignant_status"
            ]
            .astype(str)
            ==
            status,

            "cnv_burden_primary"
        ]

        .dropna()
        .values
    )


    plot_data.append(
        values
    )


fig, ax = plt.subplots(
    figsize=(6.5, 5.2)
)


bp = ax.boxplot(

    plot_data,

    tick_labels=[
        "非恶性",
        "不确定",
        "恶性"
    ],

    showfliers=False,

    patch_artist=True
)


box_colors = [
    "#4DBBD5",
    "#BDBDBD",
    "#E64B35"
]


for patch, color in zip(
    bp["boxes"],
    box_colors
):

    patch.set_facecolor(
        color
    )

    patch.set_alpha(
        0.75
    )


ax.set_ylabel(
    "CNV burden（平均绝对值）"
)

ax.set_title(
    "最终恶性判定与CNV burden"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "Step22D_03_CNV_Burden_ByMalignancy.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    FIGURES_DIR /
    "Step22D_03_CNV_Burden_ByMalignancy.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP22D - CELL 68
# CNV CHROMOSOME SUMMARY BY FINAL STATUS
# ============================================================

axes_dict = (
    cnv.pl.chromosome_heatmap_summary(

        adata_final,

        groupby=
            "malignant_status",

        use_rep=
            "cnv",

        cmap=
            "bwr",

        figsize=
            (16, 6),

        show=False
    )
)


fig = None


for ax_obj in axes_dict.values():

    if hasattr(
        ax_obj,
        "figure"
    ):

        fig = (
            ax_obj.figure
        )

        break


assert fig is not None


fig.savefig(
    FIGURES_DIR /
    "Step22D_04_CNV_ChromosomeSummary_ByMalignancy.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    FIGURES_DIR /
    "Step22D_04_CNV_ChromosomeSummary_ByMalignancy.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP22D - CELL 69
# SAVE FINAL CELL-LEVEL MALIGNANCY TABLE
# ============================================================

final_columns = [

    "sample",
    "patient",
    "site",

    "major_celltype",
    "major_cluster_candidate",

    CNV_CLUSTER_KEY,

    "cnv_reference_class",

    "cnv_burden_primary",
    "cnv_burden_balanced",
    "cnv_burden_heponly",

    "mature_hepatocyte_score",
    "ductal_progenitor_score",
    "tumor_associated_expression_score",

    "malignant_status",
    "malignancy_confidence",
    "malignancy_basis"
]


final_columns = [
    x
    for x in final_columns
    if x in adata_final.obs.columns
]


final_cell_table = (
    adata_final.obs[
        final_columns
    ]
    .copy()
)


final_cell_table.index.name = (
    "Cell"
)


FINAL_CELL_TABLE = (
    RESULTS_DIR /
    "Step22D_Final_Cell_Level_Malignancy.csv"
)


final_cell_table.to_csv(
    FINAL_CELL_TABLE,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    FINAL_CELL_TABLE
)

print(
    "Cells:",
    f"{len(final_cell_table):,}"
)

In [ ]:
# ============================================================
# STEP22D - CELL 70
# SAVE FINAL STEP22 OBJECT
# ============================================================

FINAL_STEP22_H5AD = (
    DATA_DIR /
    "GSE149614_STEP22_FINAL_MALIGNANCY_ANNOTATED.h5ad"
)


adata_final.write_h5ad(
    FINAL_STEP22_H5AD,
    compression="gzip"
)


print("=" * 100)
print("STEP22 FINAL H5AD SAVED")
print("=" * 100)

print(
    FINAL_STEP22_H5AD
)

print(
    "\nShape:",
    adata_final.shape
)

print(
    "\nMalignancy status:"
)

display(
    adata_final.obs[
        "malignant_status"
    ]
    .value_counts(
        sort=False
    )
)

In [ ]:
# ============================================================
# STEP22D - CELL 71
# FINAL STEP22 DECISION RECORD
# ============================================================

final_counts_dict = {

    str(k):
        int(v)

    for k, v in (

        adata_final.obs[
            "malignant_status"
        ]
        .value_counts(
            sort=False
        )
        .items()
    )
}


step22_final_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step22_Malignant_Cell_Identification",

    "Status":
        "COMPLETE",

    "Input_epithelial_cells":
        int(
            adata_final.n_obs
        ),

    "CNV_method":
        "infercnvpy",

    "CNV_primary_reference": [
        "Normal-site Hepatocyte",
        "Normal-site Ductal_Epithelial"
    ],

    "Primary_CNV_clustering":
        "Leiden resolution 0.4",

    "Primary_reference_sensitivity":
        {
            "Primary_vs_patient_balanced_burden_rho":
                0.99782,

            "Primary_vs_patient_balanced_cluster_ARI":
                0.9942,

            "Primary_vs_hepatocyte_only_burden_rho":
                0.91393,

            "Primary_vs_hepatocyte_only_cluster_ARI":
                0.9850
        },

    "Final_labels": [
        "Malignant",
        "Non_malignant",
        "Ambiguous"
    ],

    "Final_counts":
        final_counts_dict,

    "High_confidence_malignant_logic":
        (
            "Atypical epithelial-dominant, tumor-associated, "
            "normal-reference-depleted patient-specific CNV clone."
        ),

    "Tumor_hepatocyte_logic":
        (
            "Tumor-site mature-hepatocyte CNV clusters were compared "
            "with patient-matched normal hepatocytes when >=20 normal "
            "cells were available; otherwise pooled normal hepatocytes "
            "were used."
        ),

    "Conservative_policy":
        (
            "Cells with conflicting CNV/tissue/phenotype evidence "
            "were retained as Ambiguous rather than forced into a "
            "binary malignant/non-malignant classification."
        ),

    "Important_interpretation":
        (
            "Malignancy is inferred from scRNA-seq expression-derived "
            "CNV and transcriptional/contextual evidence, not directly "
            "measured by DNA sequencing."
        ),

    "Final_h5ad":
        str(
            FINAL_STEP22_H5AD
        ),

    "Next_step":
        (
            "Step23_Malignant_Cell_State_Analysis"
        )
}


SUMMARY_PATH = (
    LOGS_DIR /
    "Step22_Final_Summary.json"
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step22_final_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    json.dumps(
        step22_final_summary,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP22D - CELL 72
# STEP22 OUTPUT MANIFEST
# ============================================================

manifest_rows = []


folders = [

    (
        "Data",
        DATA_DIR
    ),

    (
        "Results",
        RESULTS_DIR
    ),

    (
        "Figures",
        FIGURES_DIR
    ),

    (
        "Logs",
        LOGS_DIR
    ),

    (
        "Reference",
        ROOT / "Reference"
    )
]


for category, folder in folders:

    if not folder.exists():
        continue

    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (
    RESULTS_DIR /
    "Step22_Output_Manifest.csv"
)


manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 100)
print("STEP22 OUTPUT MANIFEST")
print("=" * 100)

display(
    manifest_df
)

print(
    "\nSaved:",
    MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP22D - CELL 73
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 115)
print("STEP22 FINAL REVIEW")
print("=" * 115)


print("\n[1] Paired-normal CNV evidence")
print("-" * 115)

display(
    paired_cnv_evidence
    .round(5)
)


print("\n[2] Final CNV cluster calls")
print("-" * 115)

display(

    cluster_final[
        [
            "CNV_cluster",
            "N_cells",
            "Structural_class",
            "Median_CNV",
            "Dominant_patient",
            "Final_cluster_call",
            "Call_reason"
        ]
    ]
)


print("\n[3] Final cell-level malignancy")
print("-" * 115)

display(
    final_status_summary
    .round(2)
)


print("\n[4] Malignancy × Site")
print("-" * 115)

display(
    status_site
)


print("\n[5] Malignancy × Patient")
print("-" * 115)

display(
    status_patient
)


print("\n[6] Normal-site atypical epithelial")
print("-" * 115)

print(
    "N =",
    len(
        normal_atypical
    )
)

display(
    normal_atypical[
        "malignant_status"
    ]
    .value_counts()
)


print("\n[7] Final files")
print("-" * 115)

for p in [

    FINAL_STEP22_H5AD,

    FINAL_CELL_TABLE,

    RESULTS_DIR /
    "Step22D_Final_CNV_Cluster_Call.csv",

    RESULTS_DIR /
    "Step22D_TumorHepatocyte_PairedNormal_CNV.csv",

    SUMMARY_PATH,

    MANIFEST_PATH

]:

    print(
        "[OK]"
        if p.exists()
        else "[MISSING]",
        p.name
    )


print("\n" + "=" * 115)

print(
    "STEP22 STATUS = COMPLETE"
)

print(
    "NEXT = Step23 malignant-cell state analysis"
)

print("=" * 115)